# Freddie Mac Mortgage PD — Data Ingestion

**Objective.** Transform Freddie Mac single-family loan-level origination and performance files into a modeling-ready dataset with a clearly defined 36-month default target.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

## 1. Origination data

Load and audit origination characteristics, data types, missingness, and categorical fields.


In [ ]:
import zipfile
from pathlib import Path

zip_path = Path(r"./data/raw/sample_2018.zip")

with zipfile.ZipFile(zip_path, "r") as z:
    print("Files inside the ZIP:")
    for name in z.namelist():
        print(name)

In [ ]:
import zipfile
import pandas as pd

zip_path = r"./data/raw/sample_2018.zip"

with zipfile.ZipFile(zip_path, "r") as z:
    with z.open("sample_orig_2018.txt") as f:
        orig_2018 = pd.read_csv(
            f,
            sep="|",
            header=None,
            low_memory=False
        )

print("Shape:", orig_2018.shape)
print("\nFirst 5 rows:")
display(orig_2018.head())

In [ ]:
print("Number of columns:", orig_2018.shape[1])

for i in range(orig_2018.shape[1]):
    print(i, "→", orig_2018.iloc[0, i])

In [ ]:
orig_columns = [
    "CreditScore",
    "FirstPaymentDate",
    "FirstTimeHomebuyerFlag",
    "MaturityDate",
    "MSA",
    "MIPercent",
    "NumberOfUnits",
    "OccupancyStatus",
    "OriginalCLTV",
    "OriginalDTI",
    "OriginalUPB",
    "OriginalLTV",
    "OriginalInterestRate",
    "Channel",
    "PPMFlag",
    "AmortizationType",
    "PropertyState",
    "PropertyType",
    "PostalCode",
    "LoanSequenceNumber",
    "LoanPurpose",
    "OriginalLoanTerm",
    "NumberOfBorrowers",
    "SellerName",
    "ServicerName",
    "SuperConformingFlag",
    "PreReliefRefinanceLoanSequenceNumber",
    "SpecialEligibilityProgram",
    "ReliefRefinanceIndicator",
    "PropertyValuationMethod",
    "InterestOnlyIndicator"
]

orig_2018.columns = orig_columns

print("Number of columns:", len(orig_2018.columns))
print(orig_2018.columns.tolist())

In [ ]:
orig_2018.info()

In [ ]:
orig_2018.isna().sum().sort_values(ascending=False)

In [ ]:
orig_2018.shape

In [ ]:
categorical_cols = [
    "FirstTimeHomebuyerFlag",
    "OccupancyStatus",
    "Channel",
    "PPMFlag",
    "AmortizationType",
    "PropertyType",
    "LoanPurpose",
    "SpecialEligibilityProgram",
    "PropertyValuationMethod"
]

for col in categorical_cols:
    print(f"\n--- {col} ---")
    print(orig_2018[col].value_counts(dropna=False))

In [ ]:
numeric_cols = [
    "CreditScore",
    "OriginalCLTV",
    "OriginalDTI",
    "OriginalUPB",
    "OriginalLTV",
    "OriginalInterestRate",
    "OriginalLoanTerm",
    "NumberOfBorrowers",
    "NumberOfUnits",
    "MIPercent"
]

orig_2018[numeric_cols].describe().T

In [ ]:
for col in numeric_cols:
    print(f"\n--- {col} ---")
    print("Unique values:", orig_2018[col].nunique())
    print("Min:", orig_2018[col].min())
    print("Max:", orig_2018[col].max())

In [ ]:
special_checks = {
    "CreditScore": [9999],
    "OriginalCLTV": [999],
    "OriginalDTI": [999],
    "OriginalLTV": [999]
}

for col, values in special_checks.items():
    print(f"\n--- {col} ---")
    for value in values:
        print(f"{value}: {(orig_2018[col] == value).sum()} observations")

In [ ]:
with zipfile.ZipFile(zip_path, "r") as z:
    with z.open("sample_perf_2018.txt") as f:
        perf_2018 = pd.read_csv(
            f,
            sep="|",
            header=None,
            low_memory=False
        )

print("Shape:", perf_2018.shape)
display(perf_2018.head())

In [ ]:
print("Number of columns:", perf_2018.shape[1])

for i in range(perf_2018.shape[1]):
    print(i, "→", perf_2018.iloc[0, i])

## 2. Performance data and default-event construction

Performance histories are used to identify delinquency/default events within the intended seasoning window.


In [ ]:
perf_columns = [
    "LoanSequenceNumber",
    "MonthlyReportingPeriod",
    "CurrentActualUPB",
    "CurrentLoanDelinquencyStatus",
    "LoanAge",
    "RemainingMonthsToLegalMaturity",
    "DefectSettlementDate",
    "ModificationFlag",
    "ZeroBalanceCode",
    "ZeroBalanceEffectiveDate",
    "CurrentInterestRate",
    "CurrentDeferredUPB",
    "DueDateOfLastPaidInstallment",
    "MIRecoveries",
    "NetSaleProceeds",
    "NonMIRecoveries",
    "Expenses",
    "LegalCosts",
    "MaintenanceAndPreservationCosts",
    "TaxesAndInsurance",
    "MiscellaneousExpenses",
    "ActualLossCalculation",
    "ModificationCost",
    "StepModificationFlag",
    "DeferredPaymentPlan",
    "EstimatedLoanToValue",
    "ZeroBalanceRemovalUPB",
    "DelinquentAccruedInterest",
    "DelinquencyDueToDisaster",
    "BorrowerAssistanceStatusCode",
    "CurrentMonthModificationCost",
    "InterestBearingUPB",
    "DeferredPaymentPlanUPB",
    "MIType",
    "NonMIInsuranceType"
]

perf_2018.columns = perf_columns

print("Number of columns:", len(perf_2018.columns))
print(perf_2018.columns.tolist())

In [ ]:
perf_2018[[
    "LoanSequenceNumber",
    "MonthlyReportingPeriod",
    "CurrentActualUPB",
    "CurrentLoanDelinquencyStatus",
    "LoanAge",
    "RemainingMonthsToLegalMaturity",
    "ZeroBalanceCode",
    "ZeroBalanceEffectiveDate",
    "ActualLossCalculation",
    "ZeroBalanceRemovalUPB"
]].head(10)

In [ ]:
print("Delinquency status:")
print(perf_2018["CurrentLoanDelinquencyStatus"].value_counts(dropna=False).sort_index())

print("\nZero balance codes:")
print(perf_2018["ZeroBalanceCode"].value_counts(dropna=False).sort_index())

In [ ]:
for code in sorted(perf_2018["ZeroBalanceCode"].dropna().unique()):
    subset = perf_2018[perf_2018["ZeroBalanceCode"] == code]
    
    print(f"\nZeroBalanceCode = {int(code)}")
    print("Observations:", len(subset))
    print("Loans:", subset["LoanSequenceNumber"].nunique())
    print("Average delinquency:", 
          pd.to_numeric(
              subset["CurrentLoanDelinquencyStatus"], 
              errors="coerce"
          ).mean())

In [ ]:
default_candidate_codes = [2, 3, 9, 15, 16, 96]

candidate_loans = perf_2018[
    perf_2018["ZeroBalanceCode"].isin(default_candidate_codes)
]["LoanSequenceNumber"].unique()

print("Number of candidate loans:", len(candidate_loans))

for loan in candidate_loans[:10]:
    loan_history = perf_2018[
        perf_2018["LoanSequenceNumber"] == loan
    ][[
        "LoanSequenceNumber",
        "MonthlyReportingPeriod",
        "CurrentLoanDelinquencyStatus",
        "CurrentActualUPB",
        "ZeroBalanceCode",
        "ZeroBalanceEffectiveDate",
        "ZeroBalanceRemovalUPB"
    ]]
    
    print("\n", loan)
    display(loan_history.tail(10))

In [ ]:
# Convert delinquency status to numeric.
# "RA" and other non-numeric statuses become NaN.
perf_2018["DelinqNumeric"] = pd.to_numeric(
    perf_2018["CurrentLoanDelinquencyStatus"],
    errors="coerce"
)

# First 12 months of performance for each loan
first_12m = perf_2018[
    perf_2018["LoanAge"].between(0, 11)
]

# Loans that ever reached 90+ days delinquent in those 12 months
default_12m_loans = first_12m.loc[
    first_12m["DelinqNumeric"] >= 3,
    "LoanSequenceNumber"
].nunique()

total_loans = orig_2018["LoanSequenceNumber"].nunique()

print("Total 2018 loans:", total_loans)
print("12-month 90+ DPD events:", default_12m_loans)
print("12-month default rate:", default_12m_loans / total_loans)

In [ ]:
# Distribution of first 90+ DPD occurrence
default_events = first_12m[
    first_12m["DelinqNumeric"] >= 3
].copy()

first_default_month = (
    default_events
    .groupby("LoanSequenceNumber")["LoanAge"]
    .min()
)

print("Unique defaulting loans:", first_default_month.nunique())

print("\nFirst-default month distribution:")
print(first_default_month.value_counts().sort_index())

print("\nSummary:")
print(first_default_month.describe())

In [ ]:
first_12m[first_12m["DelinqNumeric"] >= 3]

In [ ]:
# Unique loans that ever reach 90+ days delinquent
ever_90dpd = perf_2018[
    perf_2018["DelinqNumeric"] >= 3
]["LoanSequenceNumber"].nunique()

# Unique loans that ever reach 60+ days delinquent
ever_60dpd = perf_2018[
    perf_2018["DelinqNumeric"] >= 2
]["LoanSequenceNumber"].nunique()

# Unique loans that ever reach 30+ days delinquent
ever_30dpd = perf_2018[
    perf_2018["DelinqNumeric"] >= 1
]["LoanSequenceNumber"].nunique()

total_loans = orig_2018["LoanSequenceNumber"].nunique()

print("Total loans:", total_loans)

print("\nEver 30+ DPD:", ever_30dpd,
      "Rate:", ever_30dpd / total_loans)

print("Ever 60+ DPD:", ever_60dpd,
      "Rate:", ever_60dpd / total_loans)

print("Ever 90+ DPD:", ever_90dpd,
      "Rate:", ever_90dpd / total_loans)

In [ ]:
import zipfile
import os

zip_path = r"./data/raw/sample_2022.zip"

with zipfile.ZipFile(zip_path, "r") as z:
    print("Files inside the ZIP:")
    for f in z.namelist():
        print(f)

In [ ]:
import zipfile
import os

zip_path = r"./data/raw/sample_2022.zip"

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(r"./data/raw/sample_2022")

print("Extracted files:")
print(os.listdir(r"./data/raw/sample_2022"))

In [ ]:
import pandas as pd

perf_2022 = pd.read_csv(
    r"./data/raw/sample_2022/sample_perf_2022.txt",
    sep="|",
    header=None
)

print("Shape:", perf_2022.shape)
print("Maximum LoanAge:", perf_2022[4].max())
print("Minimum LoanAge:", perf_2022[4].min())

In [ ]:
loans_36m = perf_2022.loc[
    perf_2022[4] >= 36,
    0
].nunique()

total_2022_loans = perf_2022[0].nunique()

print("Total 2022 loans:", total_2022_loans)
print("Loans observed through month 36:", loans_36m)
print("Share observed through month 36:", loans_36m / total_2022_loans)

In [ ]:
# Convert delinquency status to numeric
perf_2022["DelinqNumeric"] = pd.to_numeric(
    perf_2022[3],
    errors="coerce"
)

# 90+ DPD within first 36 months
default_36m_2022 = perf_2022[
    (perf_2022[4] <= 35) &
    (perf_2022["DelinqNumeric"] >= 3)
]["LoanSequenceNumber" if "LoanSequenceNumber" in perf_2022.columns else 0].nunique()

print("2022 loans reaching 90+ DPD within 36 months:", default_36m_2022)

In [ ]:
default_36m_2022 = perf_2022[
    (perf_2022[4] <= 35) &
    (perf_2022["DelinqNumeric"] >= 3)
][0].nunique()

print("2022 loans reaching 90+ DPD within 36 months:", default_36m_2022)

In [ ]:
# 36-month default rate among loans with full 36-month observation

default_rate_36m_2022 = default_36m_2022 / loans_36m

print("2022 36-month default rate:", default_rate_36m_2022 / loans_36m)

In [ ]:
# Recalculate the denominator from scratch
loans_36m_2022 = perf_2022.loc[
    perf_2022[4] >= 35,
    "LoanSequenceNumber"
].nunique()

# Recalculate the number of 90+ DPD loans from scratch
default_36m_2022 = perf_2022.loc[
    (perf_2022[4] <= 35) &
    (perf_2022["DelinqNumeric"] >= 3),
    "LoanSequenceNumber"
].nunique()

# Calculate rate
default_rate_36m_2022 = default_36m_2022 / loans_36m_2022

print("Loans observed through month 36:", loans_36m_2022)
print("90+ DPD loans within 36 months:", default_36m_2022)
print("36-month default rate:", default_rate_36m_2022)
print("36-month default rate (%):", default_rate_36m_2022 * 100)

In [ ]:
# Recalculate 2022 36-month denominator and default count

loans_36m_2022 = perf_2022.loc[
    perf_2022[4] >= 35,
    0
].nunique()

default_36m_2022 = perf_2022.loc[
    (perf_2022[4] <= 35) &
    (perf_2022["DelinqNumeric"] >= 3),
    0
].nunique()

default_rate_36m_2022 = default_36m_2022 / loans_36m_2022

print("Loans observed through month 36:", loans_36m_2022)
print("90+ DPD loans within 36 months:", default_36m_2022)
print("36-month default rate:", default_rate_36m_2022)
print("36-month default rate (%):", default_rate_36m_2022 * 100)

## 3. Corrected 36-month target construction

The target is restricted to loans with a sufficiently complete observation window, avoiding right-censoring in the default label.


In [ ]:
# Create 36-month default target for 2022 cohort

# Loans that have a full 36-month observation window
eligible_2022 = perf_2022.loc[
    perf_2022[4] >= 35,
    0
].unique()

# Loans that reached 90+ DPD within first 36 months
default_loans_2022 = perf_2022.loc[
    (perf_2022[4] <= 35) &
    (perf_2022["DelinqNumeric"] >= 3),
    0
].unique()

# Create loan-level target
target_2022 = pd.DataFrame({
    "LoanSequenceNumber": eligible_2022
})

target_2022["Default_36M"] = (
    target_2022["LoanSequenceNumber"]
    .isin(default_loans_2022)
    .astype(int)
)

print("Target dataset shape:", target_2022.shape)
print("\nTarget distribution:")
print(target_2022["Default_36M"].value_counts())

print("\nTarget proportions:")
print(target_2022["Default_36M"].value_counts(normalize=True))

In [ ]:
# Verify the 2022 default target

check_defaults = perf_2022[
    (perf_2022[4] <= 35) &
    (perf_2022["DelinqNumeric"] >= 3)
][0].nunique()

print("Unique 90+ DPD loans:", check_defaults)
print("Targeted defaults:", target_2022["Default_36M"].sum())

print("\nDifference:",
      check_defaults - target_2022["Default_36M"].sum())

In [ ]:
# Find the 552 defaulting loans that were excluded from target_2022

default_loans = set(
    perf_2022.loc[
        (perf_2022[4] <= 35) &
        (perf_2022["DelinqNumeric"] >= 3),
        0
    ].unique()
)

target_loans = set(target_2022["LoanSequenceNumber"])

excluded_defaults = default_loans - target_loans

print("Defaulting loans:", len(default_loans))
print("Excluded from target:", len(excluded_defaults))

# Inspect their maximum observed LoanAge
excluded_perf = perf_2022[
    perf_2022[0].isin(excluded_defaults)
]

print("\nMaximum LoanAge among excluded defaulting loans:")
print(excluded_perf.groupby(0)[4].max().describe())

print("\nFirst 10 excluded loans:")
print(
    excluded_perf[
        [0, 1, 3, 4, "DelinqNumeric"]
    ]
    .sort_values([0, 4])
    .head(30)
)

In [ ]:
# Identify all unique loans with a 90+ DPD event within 36 months

default_loans_2022 = set(
    perf_2022.loc[
        (perf_2022[4] <= 35) &
        (perf_2022["DelinqNumeric"] >= 3),
        0
    ].unique()
)

# Identify loans that reach the full 36-month observation window
full_window_loans_2022 = set(
    perf_2022.loc[
        perf_2022[4] >= 35,
        0
    ].unique()
)

# Non-default loans must have a full window AND no 90+ DPD
nondefault_loans_2022 = (
    full_window_loans_2022 - default_loans_2022
)

print("Unique default loans:", len(default_loans_2022))
print("Full-window loans:", len(full_window_loans_2022))
print("Eligible non-default loans:", len(nondefault_loans_2022))

print("\nCorrected eligible sample:",
      len(default_loans_2022) + len(nondefault_loans_2022))

In [ ]:
# Build corrected 36-month target for 2022

target_2022 = pd.DataFrame({
    "LoanSequenceNumber": list(
        default_loans_2022 | nondefault_loans_2022
    )
})

target_2022["Default_36M"] = (
    target_2022["LoanSequenceNumber"]
    .isin(default_loans_2022)
    .astype(int)
)

print("Final target dataset shape:", target_2022.shape)

print("\nTarget distribution:")
print(target_2022["Default_36M"].value_counts())

print("\nTarget proportions:")
print(target_2022["Default_36M"].value_counts(normalize=True))

print("\nDefault rate:",
      target_2022["Default_36M"].mean())

In [ ]:
# First, check the origination dataframe name and shape

print("Origination shape:", orig_2022.shape)
print("Target shape:", target_2022.shape)

print("\nOrigination columns:")
print(orig_2022.columns.tolist())

In [ ]:
import zipfile
import pandas as pd

zip_path = r"./data/raw/sample_2022.zip"

with zipfile.ZipFile(zip_path, "r") as z:
    with z.open("sample_orig_2022.txt") as f:
        orig_2022 = pd.read_csv(
            f,
            sep="|",
            header=None
        )

print("Origination shape:", orig_2022.shape)
print("Number of columns:", len(orig_2022.columns))

In [ ]:
orig_columns = [
    'CreditScore',
    'FirstPaymentDate',
    'FirstTimeHomebuyerFlag',
    'MaturityDate',
    'MSA',
    'MIPercent',
    'NumberOfUnits',
    'OccupancyStatus',
    'OriginalCLTV',
    'OriginalDTI',
    'OriginalUPB',
    'OriginalLTV',
    'OriginalInterestRate',
    'Channel',
    'PPMFlag',
    'AmortizationType',
    'PropertyState',
    'PropertyType',
    'PostalCode',
    'LoanSequenceNumber',
    'LoanPurpose',
    'OriginalLoanTerm',
    'NumberOfBorrowers',
    'SellerName',
    'ServicerName',
    'SuperConformingFlag',
    'PreReliefRefinanceLoanSequenceNumber',
    'SpecialEligibilityProgram',
    'ReliefRefinanceIndicator',
    'PropertyValuationMethod',
    'InterestOnlyIndicator'
]

orig_2022.columns = orig_columns

print(orig_2022.shape)
print(orig_2022.columns.tolist())

## 4. Merge origination characteristics with the target

Construct the loan-level modeling sample and validate the one-to-one merge.


In [ ]:
# Merge origination characteristics with the 36-month target

model_2022 = target_2022.merge(
    orig_2022,
    on="LoanSequenceNumber",
    how="left",
    validate="one_to_one"
)

print("Model dataset shape:", model_2022.shape)

print("\nTarget distribution:")
print(model_2022["Default_36M"].value_counts())

print("\nMissing target values:")
print(model_2022["Default_36M"].isna().sum())

In [ ]:
# Check missing values in the modelling dataset

missing_2022 = model_2022.isna().sum()

missing_2022 = missing_2022[missing_2022 > 0].sort_values(ascending=False)

print("Variables with missing values:")
print(missing_2022)

print("\nTotal variables with missing values:", len(missing_2022))

In [ ]:
# Drop variables that are unsuitable because of missingness

drop_cols = [
    "SuperConformingFlag",
    "PreReliefRefinanceLoanSequenceNumber",
    "MSA"
]

model_2022_clean = model_2022.drop(columns=drop_cols)

print("Clean dataset shape:", model_2022_clean.shape)

print("\nRemaining missing values:")
print(model_2022_clean.isna().sum().sort_values(ascending=False).head(10))

In [ ]:
print(model_2022_clean.columns.tolist())

In [ ]:
%whos DataFrame

In [ ]:
# Copy the cleaned modelling dataset
model_df = model_2022_clean.copy()

# Identify ID and target
id_col = "LoanSequenceNumber"
target_col = "Default_36M"

# Separate predictors and target
X = model_df.drop(columns=[id_col, target_col])
y = model_df[target_col]

print("Model dataset shape:", model_df.shape)
print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nTarget distribution:")
print(y.value_counts())

print("\nTarget proportions:")
print(y.value_counts(normalize=True))

In [ ]:
# List all predictors with their data types and number of unique values

predictor_summary = pd.DataFrame({
    "dtype": X.dtypes,
    "unique_values": X.nunique(),
    "missing_values": X.isna().sum()
}).sort_values("unique_values")

print(predictor_summary.to_string())

In [ ]:
# Show the actual unique values for low-cardinality categorical variables

for col in X.columns:
    if X[col].nunique() <= 15:
        print(f"\n--- {col} ---")
        print(X[col].value_counts(dropna=False).head(20))

In [ ]:
print("ServicerName values:")
print(model_2022_clean["ServicerName"].value_counts(dropna=False))

print("\nNumber of unique ServicerName values:")
print(model_2022_clean["ServicerName"].nunique())

print("\nData type:")
print(model_2022_clean["ServicerName"].dtype)

In [ ]:
print("SellerName values:")
print(model_2022_clean["SellerName"].value_counts(dropna=False))

print("\nNumber of unique SellerName values:")
print(model_2022_clean["SellerName"].nunique())

print("\nData type:")
print(model_2022_clean["SellerName"].dtype)

In [ ]:
print("ReliefRefinanceIndicator values:")
print(model_2022_clean["ReliefRefinanceIndicator"].value_counts(dropna=False))

print("\nNumber of unique values:")
print(model_2022_clean["ReliefRefinanceIndicator"].nunique())

print("\nData type:")
print(model_2022_clean["ReliefRefinanceIndicator"].dtype)

In [ ]:
print("FirstPaymentDate range:")
print("Minimum:", model_2022_clean["FirstPaymentDate"].min())
print("Maximum:", model_2022_clean["FirstPaymentDate"].max())

print("\nUnique values:")
print(sorted(model_2022_clean["FirstPaymentDate"].unique()))

print("\nFrequency:")
print(model_2022_clean["FirstPaymentDate"].value_counts().sort_index())

In [ ]:
print("MaturityDate range:")
print("Minimum:", model_2022_clean["MaturityDate"].min())
print("Maximum:", model_2022_clean["MaturityDate"].max())

print("\nNumber of unique values:")
print(model_2022_clean["MaturityDate"].nunique())

print("\nFirst 20 values:")
print(sorted(model_2022_clean["MaturityDate"].unique())[:20])

print("\nLast 20 values:")
print(sorted(model_2022_clean["MaturityDate"].unique())[-20:])

In [ ]:
print("PostalCode dtype:")
print(model_2022_clean["PostalCode"].dtype)

print("\nNumber of unique PostalCodes:")
print(model_2022_clean["PostalCode"].nunique())

print("\nMinimum:")
print(model_2022_clean["PostalCode"].min())

print("\nMaximum:")
print(model_2022_clean["PostalCode"].max())

print("\nSample values:")
print(model_2022_clean["PostalCode"].head(30).tolist())

In [ ]:
constant_cols = [
    "PPMFlag",
    "AmortizationType",
    "PropertyValuationMethod",
    "SpecialEligibilityProgram",
    "InterestOnlyIndicator"
]

for col in constant_cols:
    print(f"\n--- {col} ---")
    print("Unique values:", model_2022_clean[col].unique())
    print("Number of unique values:", model_2022_clean[col].nunique())

In [ ]:
numeric_check_cols = [
    "CreditScore",
    "OriginalCLTV",
    "OriginalDTI",
    "OriginalUPB",
    "OriginalLTV",
    "OriginalInterestRate",
    "OriginalLoanTerm",
    "NumberOfBorrowers",
    "NumberOfUnits",
    "MIPercent"
]

for col in numeric_check_cols:
    print(f"\n--- {col} ---")
    print("Min:", model_2022_clean[col].min())
    print("Max:", model_2022_clean[col].max())
    print("Unique:", model_2022_clean[col].nunique())

## 5. Data cleaning and feature preparation

Handle sentinel values, derive time-based features, remove unsuitable identifiers/constants, and inspect missingness before model fitting.


In [ ]:
print("Final modelling dataset:")
print("Rows:", model_2022_clean.shape[0])
print("Columns:", model_2022_clean.shape[1])

print("\nMissing values:")
print(model_2022_clean.isna().sum().sort_values(ascending=False).head(20))

print("\nTarget:")
print(model_2022_clean["Default_36M"].value_counts())

print("\nTarget rate:")
print(model_2022_clean["Default_36M"].mean())

In [ ]:
model_df = model_2022_clean.copy()

print("Starting shape:", model_df.shape)

In [ ]:
import numpy as np

model_df["CreditScore"] = model_df["CreditScore"].replace(9999, np.nan)
model_df["OriginalDTI"] = model_df["OriginalDTI"].replace(999, np.nan)

print("Special codes after replacement:")

print("CreditScore 9999:",
      (model_df["CreditScore"] == 9999).sum())

print("OriginalDTI 999:",
      (model_df["OriginalDTI"] == 999).sum())

In [ ]:
model_df["FirstPaymentYear"] = (
    model_df["FirstPaymentDate"] // 100
).astype(int)

model_df["FirstPaymentMonth"] = (
    model_df["FirstPaymentDate"] % 100
).astype(int)

print(
    model_df[
        ["FirstPaymentDate", "FirstPaymentYear", "FirstPaymentMonth"]
    ].head(10)
)

In [ ]:
model_df["MaturityYear"] = (
    model_df["MaturityDate"] // 100
).astype(int)

model_df["MaturityMonth"] = (
    model_df["MaturityDate"] % 100
).astype(int)

print(
    model_df[
        ["MaturityDate", "MaturityYear", "MaturityMonth"]
    ].head(10)
)

In [ ]:
model_df["ScheduledTermMonths"] = (
    (model_df["MaturityYear"] - model_df["FirstPaymentYear"]) * 12
    + (model_df["MaturityMonth"] - model_df["FirstPaymentMonth"])
)

print(model_df["ScheduledTermMonths"].describe())

In [ ]:
model_df = model_df.drop(
    columns=["FirstPaymentDate", "MaturityDate"]
)

print("Shape after date transformation:", model_df.shape)

In [ ]:
drop_cols = [
    "PPMFlag",
    "AmortizationType",
    "PropertyValuationMethod",
    "SpecialEligibilityProgram",
    "InterestOnlyIndicator",
    "PostalCode"
]

model_df = model_df.drop(columns=drop_cols)

print("Shape after dropping constants + PostalCode:", model_df.shape)

In [ ]:
print("Remaining columns:")
for i, col in enumerate(model_df.columns):
    print(i, "→", col)

In [ ]:
print("\nData types:")
print(model_df.dtypes)

In [ ]:
print("\nMissing values:")
print(
    model_df.isna().sum()
    .sort_values(ascending=False)
)

In [ ]:
import numpy as np
import pandas as pd

model_df_clean = model_df.copy()

# Check sentinel values BEFORE changing anything
print("CreditScore = 9999:", 
      (model_df_clean["CreditScore"] == 9999).sum())

print("OriginalDTI = 999:", 
      (model_df_clean["OriginalDTI"] == 999).sum())

In [ ]:
# Check the actual rows with missing values

missing_rows = model_df[
    model_df["CreditScore"].isna() |
    model_df["OriginalDTI"].isna()
]

print("Number of rows with missing values:", len(missing_rows))

print("\nMissing-value pattern:")
print(
    missing_rows[["LoanSequenceNumber", "Default_36M",
                  "CreditScore", "OriginalDTI"]]
)

In [ ]:
# Check whether missingness is related to the target

print("\nDefault distribution among missing-value rows:")
print(missing_rows["Default_36M"].value_counts())

print("\nDefault rate among missing-value rows:")
print(missing_rows["Default_36M"].mean())

print("\nOverall default rate:")
print(model_df["Default_36M"].mean())

## 6. Train/test preprocessing

Fit imputation and encoding transformations on training data only, then apply the fitted transformations to held-out observations.


In [ ]:
from sklearn.model_selection import train_test_split

# Separate ID, target, and predictors
id_col = "LoanSequenceNumber"
target_col = "Default_36M"

X = model_df_clean.drop(columns=[id_col, target_col])
y = model_df_clean[target_col]

# Stratified 80/20 train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

print("\ny_train distribution:")
print(y_train.value_counts())

print("\ny_test distribution:")
print(y_test.value_counts())

print("\ny_train default rate:", y_train.mean())
print("y_test default rate:", y_test.mean())

In [ ]:
# ============================================================
# STEP 2: Identify numerical and categorical variables
# ============================================================

numeric_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

print("Numerical variables:")
print(numeric_features)

print("\nNumber of numerical variables:", len(numeric_features))

print("\nCategorical variables:")
print(categorical_features)

print("\nNumber of categorical variables:", len(categorical_features))

In [ ]:
# Check whether any numerical variables have missing values
print("\nMissing numerical values in TRAIN:")
print(
    X_train[numeric_features]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

# Check categorical missing values in TRAIN
print("\nMissing categorical values in TRAIN:")
print(
    X_train[categorical_features]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

# Numerical preprocessing
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ]
)

# Categorical preprocessing
categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        ))
    ]
)

# Combine both
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

print("Preprocessing pipeline created successfully.")

In [ ]:
# Fit preprocessing using TRAINING data only
X_train_processed = preprocessor.fit_transform(X_train)

# Transform TEST using the already-fitted preprocessing
X_test_processed = preprocessor.transform(X_test)

print("X_train processed shape:", X_train_processed.shape)
print("X_test processed shape:", X_test_processed.shape)

In [ ]:
print("Missing values in processed training data:",
      np.isnan(X_train_processed).sum())

print("Missing values in processed test data:",
      np.isnan(X_test_processed).sum())

In [ ]:
import zipfile
import os

zip_path = r"./data/raw/sample_2019.zip"
extract_path = r"./data/raw/sample_2019"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("Extracted files:")
print(os.listdir(extract_path))

In [ ]:
import pandas as pd

perf_2019 = pd.read_csv(
    os.path.join(extract_path, "sample_perf_2019.txt"),
    sep="|",
    header=None,
    low_memory=False
)

print("Shape:", perf_2019.shape)
print("Maximum LoanAge:", perf_2019[4].max())
print("Minimum LoanAge:", perf_2019[4].min())

In [ ]:
# Check 2019 loan-level 36-month observability

loans_36m_2019 = perf_2019.loc[
    perf_2019[4] >= 35,
    0
].nunique()

total_2019_loans = perf_2019[0].nunique()

print("Total 2019 loans:", total_2019_loans)
print("Loans observed through month 36:", loans_36m_2019)
print("Share observed through month 36:", loans_36m_2019 / total_2019_loans)

In [ ]:
# Check the 2019 performance structure first

print("Performance shape:", perf_2019.shape)

print("\nFirst 10 columns:")
print(perf_2019.columns[:10].tolist())

print("\nLoanAge range:")
print("Min:", perf_2019[4].min())
print("Max:", perf_2019[4].max())

In [ ]:
# Rename the important performance columns

perf_2019 = perf_2019.rename(columns={
    0: "LoanSequenceNumber",
    1: "MonthlyReportingPeriod",
    4: "LoanAge"
})

# Check whether DelinqNumeric already exists
print(perf_2019.columns.tolist())

In [ ]:
print(perf_2019.columns.tolist())

In [ ]:
# Show the first 10 columns and sample rows
print(perf_2019.iloc[:10, :10])

In [ ]:
print(perf_2019[3].value_counts(dropna=False).head(30))

In [ ]:
# Create numeric delinquency status
perf_2019["DelinqNumeric"] = pd.to_numeric(
    perf_2019[3],
    errors="coerce"
)

print(perf_2019["DelinqNumeric"].value_counts(dropna=False).sort_index())

In [ ]:
default_36m_perf_2019 = perf_2019.loc[
    (perf_2019["LoanAge"] <= 35) &
    (perf_2019["DelinqNumeric"] >= 3)
]

default_loans_2019 = default_36m_perf_2019[
    "LoanSequenceNumber"
].nunique()

print("90+ DPD loans within 36 months:", default_loans_2019)

In [ ]:
full_window_2019 = perf_2019.loc[
    perf_2019["LoanAge"] >= 35,
    "LoanSequenceNumber"
].nunique()

print("Loans observed through month 36:", full_window_2019)

default_rate_2019 = default_loans_2019 / full_window_2019

print("2019 36-month default rate:", default_rate_2019)
print("2019 36-month default rate (%):", default_rate_2019 * 100)

In [ ]:
# Full-window 2019 loans
full_window_loans_2019 = set(
    perf_2019.loc[
        perf_2019["LoanAge"] >= 35,
        "LoanSequenceNumber"
    ].unique()
)

# All loans with 90+ DPD within first 36 months
default_loans_all_2019 = set(
    perf_2019.loc[
        (perf_2019["LoanAge"] <= 35) &
        (perf_2019["DelinqNumeric"] >= 3),
        "LoanSequenceNumber"
    ].unique()
)

# Defaults that are actually in the full-window sample
eligible_default_loans_2019 = (
    default_loans_all_2019 & full_window_loans_2019
)

print("Full-window loans:", len(full_window_loans_2019))
print("All 90+ DPD loans within 36M:", len(default_loans_all_2019))
print("Eligible 90+ DPD loans:", len(eligible_default_loans_2019))
print("Defaults excluded because not full-window:",
      len(default_loans_all_2019 - full_window_loans_2019))

In [ ]:
default_rate_2019_corrected = (
    len(eligible_default_loans_2019)
    / len(full_window_loans_2019)
)

print(
    "Corrected 2019 36-month default rate:",
    default_rate_2019_corrected
)

print(
    "Corrected 2019 36-month default rate (%):",
    default_rate_2019_corrected * 100
)

In [ ]:
# Create the final 2019 target

target_2019 = pd.DataFrame({
    "LoanSequenceNumber": list(full_window_loans_2019)
})

target_2019["Default_36M"] = (
    target_2019["LoanSequenceNumber"]
    .isin(eligible_default_loans_2019)
    .astype(int)
)

print("Target dataset shape:", target_2019.shape)

print("\nTarget distribution:")
print(target_2019["Default_36M"].value_counts())

print("\nTarget proportions:")
print(target_2019["Default_36M"].value_counts(normalize=True))

print("\nDefault rate:",
      target_2019["Default_36M"].mean())

In [ ]:
orig_2019 = pd.read_csv(
    os.path.join(extract_path, "sample_orig_2019.txt"),
    sep="|",
    header=None,
    low_memory=False
)

print("Origination shape:", orig_2019.shape)
print("Number of columns:", orig_2019.shape[1])

In [ ]:
print(orig_2019.shape)

In [ ]:
print(orig_2019.head())

In [ ]:
print("Unique loans:", orig_2019[19].nunique())

In [ ]:
orig_columns = [
    "CreditScore",
    "FirstPaymentDate",
    "FirstTimeHomebuyerFlag",
    "MaturityDate",
    "MSA",
    "MIPercent",
    "NumberOfUnits",
    "OccupancyStatus",
    "OriginalCLTV",
    "OriginalDTI",
    "OriginalUPB",
    "OriginalLTV",
    "OriginalInterestRate",
    "Channel",
    "PPMFlag",
    "AmortizationType",
    "PropertyState",
    "PostalCode",
    "LoanSequenceNumber",
    "LoanPurpose",
    "OriginalLoanTerm",
    "NumberOfBorrowers",
    "SellerName",
    "ServicerName",
    "SuperConformingFlag",
    "PreReliefRefinanceLoanSequenceNumber",
    "SpecialEligibilityProgram",
    "ReliefRefinanceIndicator",
    "PropertyValuationMethod",
    "InterestOnlyIndicator",
    "ExtraColumn"
]

print(len(orig_columns))
print(orig_2019.shape[1])

In [ ]:
for i in range(orig_2019.shape[1]):
    print(i, "=>", repr(orig_2019.iloc[0, i]))

In [ ]:
orig_columns = [
    "CreditScore",
    "FirstPaymentDate",
    "FirstTimeHomebuyerFlag",
    "MaturityDate",
    "MSA",
    "MIPercent",
    "NumberOfUnits",
    "OccupancyStatus",
    "OriginalCLTV",
    "OriginalDTI",
    "OriginalUPB",
    "OriginalLTV",
    "OriginalInterestRate",
    "Channel",
    "PPMFlag",
    "AmortizationType",
    "PropertyState",
    "PropertyType",
    "PostalCode",
    "LoanSequenceNumber",
    "LoanPurpose",
    "OriginalLoanTerm",
    "NumberOfBorrowers",
    "SellerName",
    "ServicerName",
    "SuperConformingFlag",
    "PreReliefRefinanceLoanSequenceNumber",
    "SpecialEligibilityProgram",
    "ReliefRefinanceIndicator",
    "PropertyValuationMethod",
    "InterestOnlyIndicator"
]

print("Number of names:", len(orig_columns))
print("Number of columns:", orig_2019.shape[1])

In [ ]:
orig_2019.columns = orig_columns

print(orig_2019.head())
print("\nColumns:")
print(orig_2019.columns.tolist())

In [ ]:
print(
    "Unique LoanSequenceNumber:",
    orig_2019["LoanSequenceNumber"].nunique()
)

print(
    "Total rows:",
    len(orig_2019)
)

In [ ]:
model_2019 = orig_2019.merge(
    target_2019,
    on="LoanSequenceNumber",
    how="inner",
    validate="one_to_one"
)

print("Model dataset shape:", model_2019.shape)

print("\nTarget distribution:")
print(model_2019["Default_36M"].value_counts())

print("\nTarget proportions:")
print(model_2019["Default_36M"].value_counts(normalize=True))

print("\nMissing target values:")
print(model_2019["Default_36M"].isna().sum())

In [ ]:
print("\nOriginal 2019 loans:", len(orig_2019))
print("Eligible modelling loans:", len(model_2019))
print("Excluded from modelling:", len(orig_2019) - len(model_2019))

In [ ]:
# Copy the merged 2019 modelling dataset
model_2019_clean = model_2019.copy()

print("Starting shape:", model_2019_clean.shape)

print("\nMissing values:")
print(
    model_2019_clean.isna()
    .sum()
    .sort_values(ascending=False)
)

In [ ]:
# Drop variables with excessive missingness / identifiers
drop_cols_2019 = [
    "SuperConformingFlag",
    "PreReliefRefinanceLoanSequenceNumber",
    "MSA"
]

model_2019_clean = model_2019_clean.drop(columns=drop_cols_2019)

print("Shape after dropping missing variables:", model_2019_clean.shape)

print("\nRemaining missing values:")
print(
    model_2019_clean.isna()
    .sum()
    .sort_values(ascending=False)
)

In [ ]:
# Check special missing-value codes in 2019

print("CreditScore = 9999:",
      (model_2019_clean["CreditScore"] == 9999).sum())

print("OriginalDTI = 999:",
      (model_2019_clean["OriginalDTI"] == 999).sum())

print("OriginalLTV = 999:",
      (model_2019_clean["OriginalLTV"] == 999).sum())

print("OriginalCLTV = 999:",
      (model_2019_clean["OriginalCLTV"] == 999).sum())

print("OriginalInterestRate = 999:",
      (model_2019_clean["OriginalInterestRate"] == 999).sum())

In [ ]:
# Convert Freddie Mac special missing-value codes to NaN

model_2019_clean.loc[
    model_2019_clean["CreditScore"] == 9999,
    "CreditScore"
] = np.nan

model_2019_clean.loc[
    model_2019_clean["OriginalDTI"] == 999,
    "OriginalDTI"
] = np.nan

model_2019_clean.loc[
    model_2019_clean["OriginalCLTV"] == 999,
    "OriginalCLTV"
] = np.nan

print("Missing values after converting special codes:")

print(
    model_2019_clean[
        ["CreditScore", "OriginalDTI", "OriginalCLTV"]
    ].isna().sum()
)

In [ ]:
# Inspect rows containing missing values

missing_rows_2019 = model_2019_clean[
    model_2019_clean[
        ["CreditScore", "OriginalDTI", "OriginalCLTV"]
    ].isna().any(axis=1)
]

print("Number of rows with missing values:",
      len(missing_rows_2019))

print("\nMissing-value pattern:")
print(
    missing_rows_2019[
        ["LoanSequenceNumber",
         "Default_36M",
         "CreditScore",
         "OriginalDTI",
         "OriginalCLTV"]
    ]
)

print("\nDefault distribution among missing-value rows:")
print(
    missing_rows_2019["Default_36M"].value_counts()
)

print("\nDefault rate among missing-value rows:",
      missing_rows_2019["Default_36M"].mean())

print("\nOverall default rate:",
      model_2019_clean["Default_36M"].mean())

In [ ]:
import os
import zipfile

zip_path_2020 = r"./data/raw/sample_2020.zip"

extract_path_2020 = r"./data/raw/sample_2020"

os.makedirs(extract_path_2020, exist_ok=True)

with zipfile.ZipFile(zip_path_2020, "r") as z:
    z.extractall(extract_path_2020)

print("Extracted files:")
print(os.listdir(extract_path_2020))

In [ ]:
import pandas as pd

perf_2020 = pd.read_csv(
    os.path.join(extract_path_2020, "sample_perf_2020.txt"),
    sep="|",
    header=None,
    low_memory=False
)

print("Shape:", perf_2020.shape)
print("Maximum LoanAge:", perf_2020[4].max())
print("Minimum LoanAge:", perf_2020[4].min())

In [ ]:
# 2020 loans observed through month 36

loans_36m_2020 = perf_2020.loc[
    perf_2020[4] >= 35,
    0
].nunique()

total_2020_loans = perf_2020[0].nunique()

print("Total 2020 loans:", total_2020_loans)
print("Loans observed through month 36:", loans_36m_2020)
print(
    "Share observed through month 36:",
    loans_36m_2020 / total_2020_loans
)

In [ ]:
print("Number of columns:", len(perf_2020.columns))

print("\nColumn names:")
print(perf_2020.columns.tolist())

In [ ]:
# 90+ DPD events within the first 36 months
default_36m_perf_2020 = perf_2020.loc[
    (perf_2020[4] <= 35) &
    (perf_2020[34] >= 3)
]

default_loans_2020 = default_36m_perf_2020[0].nunique()

print("90+ DPD loans within 36 months:", default_loans_2020)

In [ ]:
print(
    "Raw 2020 36-month default rate:",
    default_loans_2020 / loans_36m_2020
)

print(
    "Raw 2020 36-month default rate (%):",
    default_loans_2020 / loans_36m_2020 * 100
)

In [ ]:
# Loans with a complete 36-month observation window
full_window_loans_2020 = set(
    perf_2020.loc[
        perf_2020[4] >= 35,
        0
    ].unique()
)

# All loans that experienced 90+ DPD within months 0–35
all_default_loans_2020 = set(
    default_36m_perf_2020[0].unique()
)

# Keep only defaults belonging to full-window loans
eligible_default_loans_2020 = (
    all_default_loans_2020 &
    full_window_loans_2020
)

excluded_default_loans_2020 = (
    all_default_loans_2020 -
    full_window_loans_2020
)

print("Full-window loans:", len(full_window_loans_2020))
print("All 90+ DPD loans within 36M:", len(all_default_loans_2020))
print("Eligible 90+ DPD loans:", len(eligible_default_loans_2020))
print(
    "Defaults excluded because not full-window:",
    len(excluded_default_loans_2020)
)

print(
    "\nCorrected 2020 36-month default rate:",
    len(eligible_default_loans_2020) /
    len(full_window_loans_2020)
)

print(
    "Corrected 2020 36-month default rate (%):",
    len(eligible_default_loans_2020) /
    len(full_window_loans_2020) * 100
)

In [ ]:
# Inspect the last few columns of the 2020 performance file

print(perf_2020[[0, 1, 2, 3, 4, 32, 33, 34]].head(20))

In [ ]:
print("\nColumn 34 value counts:")
print(
    perf_2020[34]
    .value_counts(dropna=False)
    .head(30)
)

In [ ]:
print("\nColumn 3 value counts:")
print(
    perf_2020[3]
    .value_counts(dropna=False)
    .head(30)
)

In [ ]:
print("\nRows where delinquency status is 03 or higher:")
print(
    perf_2020.loc[
        perf_2020[3].astype(str).str.strip().isin(
            ["03", "04", "05", "06", "07", "08", "09",
             "10", "11", "12", "13", "14", "15", "16"]
        ),
        [0, 1, 3, 4, 34]
    ].head(20)
)

In [ ]:
# Convert delinquency status (column 3) to numeric
# "RA" and any other non-numeric values become NaN

perf_2020["DelinqNumeric"] = pd.to_numeric(
    perf_2020[3],
    errors="coerce"
)

print("DelinqNumeric value counts:")
print(
    perf_2020["DelinqNumeric"]
    .value_counts(dropna=False)
    .sort_index()
    .head(25)
)

In [ ]:
# 90+ DPD = delinquency status 03 or higher
# Only consider LoanAge 0 through 35

default_36m_perf_2020 = perf_2020.loc[
    (perf_2020[4] <= 35) &
    (perf_2020["DelinqNumeric"] >= 3)
]

all_default_loans_2020 = set(
    default_36m_perf_2020[0].unique()
)

print(
    "90+ DPD loans within 36 months:",
    len(all_default_loans_2020)
)

In [ ]:
full_window_loans_2020 = set(
    perf_2020.loc[
        perf_2020[4] >= 35,
        0
    ].unique()
)

print(
    "Loans observed through month 36:",
    len(full_window_loans_2020)
)

In [ ]:
eligible_default_loans_2020 = (
    all_default_loans_2020 &
    full_window_loans_2020
)

excluded_default_loans_2020 = (
    all_default_loans_2020 -
    full_window_loans_2020
)

print("Full-window loans:", len(full_window_loans_2020))
print("All 90+ DPD loans within 36M:", len(all_default_loans_2020))
print("Eligible 90+ DPD loans:", len(eligible_default_loans_2020))
print(
    "Defaults excluded because not full-window:",
    len(excluded_default_loans_2020)
)

In [ ]:
default_rate_2010_2020 = (
    len(eligible_default_loans_2020) /
    len(full_window_loans_2020)
)

print(
    "Corrected 2020 36-month default rate:",
    default_rate_2010_2020
)

print(
    "Corrected 2020 36-month default rate (%):",
    default_rate_2010_2020 * 100
)

In [ ]:
# Create target for all full-window loans
target_2020 = pd.DataFrame({
    "LoanSequenceNumber": list(full_window_loans_2020)
})

target_2020["Default_36M"] = (
    target_2020["LoanSequenceNumber"]
    .isin(eligible_default_loans_2020)
    .astype(int)
)

print("Target dataset shape:", target_2020.shape)

print("\nTarget distribution:")
print(target_2020["Default_36M"].value_counts())

print("\nTarget proportions:")
print(
    target_2020["Default_36M"]
    .value_counts(normalize=True)
)

print(
    "\nDefault rate:",
    target_2020["Default_36M"].mean()
)

In [ ]:
orig_2020 = pd.read_csv(
    os.path.join(
        extract_path_2020,
        "sample_orig_2020.txt"
    ),
    sep="|",
    header=None,
    low_memory=False
)

print("Origination shape:", orig_2020.shape)
print("Number of columns:", len(orig_2020.columns))

In [ ]:
print("First row of 2020 origination file:")

for i, value in enumerate(orig_2020.iloc[0]):
    print(i, "=>", repr(value))

In [ ]:
orig_columns = [
    "CreditScore",
    "FirstPaymentDate",
    "FirstTimeHomebuyerFlag",
    "MaturityDate",
    "MIPercent",
    "NumberOfUnits",
    "OccupancyStatus",
    "OriginalCLTV",
    "OriginalDTI",
    "OriginalUPB",
    "OriginalLTV",
    "OriginalInterestRate",
    "Channel",
    "PropertyState",
    "PropertyType",
    "LoanSequenceNumber",
    "LoanPurpose",
    "OriginalLoanTerm",
    "NumberOfBorrowers",
    "SellerName",
    "ServicerName",
    "SuperConformingFlag",
    "PreReliefRefinanceLoanSequenceNumber",
    "SpecialEligibilityProgram",
    "ReliefRefinanceIndicator",
    "PropertyValuationMethod",
    "InterestOnlyIndicator"
]

# IMPORTANT:
# The file has 31 columns, so add the two columns that were missing
# from the 29-variable list above.

orig_columns = [
    "CreditScore",
    "FirstPaymentDate",
    "FirstTimeHomebuyerFlag",
    "MaturityDate",
    "MIPercent",
    "NumberOfUnits",
    "OccupancyStatus",
    "OriginalCLTV",
    "OriginalDTI",
    "OriginalUPB",
    "OriginalLTV",
    "OriginalInterestRate",
    "Channel",
    "PropertyState",
    "PropertyType",
    "LoanSequenceNumber",
    "LoanPurpose",
    "OriginalLoanTerm",
    "NumberOfBorrowers",
    "SellerName",
    "ServicerName",
    "SuperConformingFlag",
    "PreReliefRefinanceLoanSequenceNumber",
    "SpecialEligibilityProgram",
    "ReliefRefinanceIndicator",
    "PropertyValuationMethod",
    "InterestOnlyIndicator",
    "PropertyState",
    "PropertyType",
    "LoanPurpose",
    "OriginalLoanTerm"
]

In [ ]:
orig_columns_2020 = [
    "CreditScore",                         # 0
    "FirstPaymentDate",                   # 1
    "FirstTimeHomebuyerFlag",             # 2
    "MaturityDate",                       # 3
    "MIPercent",                          # 4
    "NumberOfUnits",                      # 5
    "OccupancyStatus",                    # 6
    "OriginalCLTV",                       # 7
    "OriginalDTI",                        # 8
    "OriginalUPB",                       # 9
    "OriginalLTV",                       # 10
    "OriginalInterestRate",              # 11
    "Channel",                            # 12
    "PropertyState",                     # 13
    "PropertyType",                      # 14
    "LoanSequenceNumber",                # 15
    "LoanPurpose",                       # 16
    "OriginalLoanTerm",                  # 17
    "NumberOfBorrowers",                 # 18
    "SellerName",                        # 19
    "ServicerName",                      # 20
    "SuperConformingFlag",               # 21
    "PreReliefRefinanceLoanSequenceNumber", # 22
    "SpecialEligibilityProgram",          # 23
    "ReliefRefinanceIndicator",           # 24
    "PropertyValuationMethod",            # 25
    "InterestOnlyIndicator",              # 26
    "Extra_27",                           # 27
    "Extra_28",                           # 28
    "Extra_29",                           # 29
    "Extra_30"                            # 30
]

In [ ]:
print("2020 column positions:")

for i in range(31):
    print(i, "=>", repr(orig_2020.iloc[0, i]))

In [ ]:
orig_columns_2020 = [
    "CreditScore",
    "FirstPaymentDate",
    "FirstTimeHomebuyerFlag",
    "MaturityDate",
    "MIPercent",
    "NumberOfUnits",
    "OccupancyStatus",
    "OriginalCLTV",
    "OriginalDTI",
    "OriginalUPB",
    "OriginalLTV",
    "OriginalInterestRate",
    "Channel",
    "PropertyState",
    "PropertyType",
    "AmortizationType",
    "PropertyState_2",
    "PropertyType_2",
    "PostalCode",
    "LoanSequenceNumber",
    "LoanPurpose",
    "OriginalLoanTerm",
    "NumberOfBorrowers",
    "SellerName",
    "ServicerName",
    "SuperConformingFlag",
    "PreReliefRefinanceLoanSequenceNumber",
    "SpecialEligibilityProgram",
    "ReliefRefinanceIndicator",
    "PropertyValuationMethod",
    "InterestOnlyIndicator"
]

orig_2020.columns = orig_columns_2020

print("Shape:", orig_2020.shape)
print(orig_2020.columns.tolist())
print("\nUnique LoanSequenceNumber:",
      orig_2020["LoanSequenceNumber"].nunique())

In [ ]:
orig_columns_2020 = [
    "CreditScore",
    "FirstPaymentDate",
    "FirstTimeHomebuyerFlag",
    "MaturityDate",
    "MIPercent",
    "NumberOfUnits",
    "OccupancyStatus",
    "OriginalCLTV",
    "OriginalDTI",
    "OriginalUPB",
    "OriginalLTV",
    "OriginalInterestRate",
    "Channel",
    "PropertyState",
    "PropertyType",
    "AmortizationType",
    "PropertyState_2",
    "PropertyType_2",
    "PostalCode",
    "LoanSequenceNumber",
    "LoanPurpose",
    "OriginalLoanTerm",
    "NumberOfBorrowers",
    "SellerName",
    "ServicerName",
    "SuperConformingFlag",
    "PreReliefRefinanceLoanSequenceNumber",
    "SpecialEligibilityProgram",
    "ReliefRefinanceIndicator",
    "PropertyValuationMethod",
    "InterestOnlyIndicator"
]

In [ ]:
for i in range(13, 31):
    print(i, "=>", repr(orig_2020.iloc[0, i]))

In [ ]:
model_2020 = orig_2020.merge(
    target_2020,
    on="LoanSequenceNumber",
    how="inner",
    validate="one_to_one"
)

print("Model dataset shape:", model_2020.shape)
print(model_2020["Default_36M"].value_counts())

In [ ]:
print("Current model columns:")
print(model_2020.columns.tolist())

print("\nMissing values:")
print(
    model_2020.isna()
    .sum()
    .sort_values(ascending=False)
)

In [ ]:
# ============================================================
# 1. CLEAN 2020 MODEL DATASET
# ============================================================

model_2020_clean = model_2020.copy()

print("Starting shape:", model_2020_clean.shape)

# Drop columns that are completely / overwhelmingly missing
drop_cols_2020 = [
    "SuperConformingFlag",
    "PreReliefRefinanceLoanSequenceNumber"
]

model_2020_clean = model_2020_clean.drop(
    columns=drop_cols_2020
)

print("\nShape after dropping unusable columns:",
      model_2020_clean.shape)

print("\nRemaining missing values:")
print(
    model_2020_clean.isna()
    .sum()
    .sort_values(ascending=False)
    .head(20)
)

In [ ]:
# ============================================================
# 2. CHECK MIPERCENT MISSINGNESS
# ============================================================

print("Missing MIPercent:",
      model_2020_clean["MIPercent"].isna().sum())

print("Total rows:",
      len(model_2020_clean))

print("Missing MIPercent share:",
      model_2020_clean["MIPercent"].isna().mean())

print("\nTarget distribution among MIPercent-missing rows:")
print(
    model_2020_clean.loc[
        model_2020_clean["MIPercent"].isna(),
        "Default_36M"
    ].value_counts()
)

print("\nDefault rate among MIPercent-missing rows:",
      model_2020_clean.loc[
          model_2020_clean["MIPercent"].isna(),
          "Default_36M"
      ].mean())

print("\nOverall default rate:",
      model_2020_clean["Default_36M"].mean())

In [ ]:
import os
import zipfile
import pandas as pd
import numpy as np

zip_path_2021 = r"./data/raw/sample_2021.zip"
extract_path_2021 = r"./data/raw/sample_2021"

os.makedirs(extract_path_2021, exist_ok=True)

with zipfile.ZipFile(zip_path_2021, "r") as zip_ref:
    zip_ref.extractall(extract_path_2021)

print("Extracted files:")
print(os.listdir(extract_path_2021))

In [ ]:
perf_2021 = pd.read_csv(
    os.path.join(extract_path_2021, "sample_perf_2021.txt"),
    sep="|",
    header=None,
    low_memory=False
)

print("Shape:", perf_2021.shape)
print("Number of columns:", len(perf_2021.columns))
print("Maximum LoanAge:", perf_2021[4].max())
print("Minimum LoanAge:", perf_2021[4].min())

In [ ]:
perf_columns = [
    "LoanSequenceNumber",
    "MonthlyReportingPeriod",
    "CurrentActualUPB",
    "CurrentLoanDelinquencyStatus",
    "LoanAge",
    "RemainingMonthsToLegalMaturity",
    "DefectSettlementDate",
    "ModificationFlag",
    "ZeroBalanceCode",
    "ZeroBalanceEffectiveDate",
    "CurrentInterestRate",
    "CurrentDeferredUPB",
    "DueDateOfLastPaidInstallment",
    "MIRecoveries",
    "NetSalesProceeds",
    "NonMIRecoveries",
    "Expenses",
    "LegalCosts",
    "MaintenanceAndPreservationCosts",
    "TaxesAndInsurance",
    "MiscellaneousExpenses",
    "ActualLossCalculation",
    "ModificationCost",
    "StepModification",
    "DeferredPaymentPlan",
    "ELTV",
    "ZeroBalanceRemovalUpb",
    "DelinquentAccruedInterest",
    "DelinquencyDueToDisaster",
    "BorrowerAssistanceStatusCode",
    "CurrentMonthModificationCost",
    "InterestBearingUPB",
    "PrincipalForgivenessAmount",
    "MortgageInsuranceCancellationIndicator",
    "CurrentPeriodModificationLoss"
]

print("Number of names:", len(perf_columns))
print("Number of columns:", len(perf_2021.columns))

In [ ]:
perf_2021.columns = perf_columns

print(perf_2021.columns.tolist())

In [ ]:
perf_2021["DelinqNumeric"] = pd.to_numeric(
    perf_2021["CurrentLoanDelinquencyStatus"],
    errors="coerce"
)

print(perf_2021["DelinqNumeric"].value_counts(dropna=False).head(30))

In [ ]:
loans_2021 = perf_2021["LoanSequenceNumber"].nunique()

full_window_loans_2021 = set(
    perf_2021.loc[
        perf_2021["LoanAge"] >= 35,
        "LoanSequenceNumber"
    ].unique()
)

print("Total 2021 loans:", loans_2021)
print("Loans observed through month 36:",
      len(full_window_loans_2021))
print(
    "Share observed through month 36:",
    len(full_window_loans_2021) / loans_2021
)

In [ ]:
default_36m_perf_2021 = perf_2021.loc[
    (perf_2021["LoanAge"] <= 35) &
    (perf_2021["DelinqNumeric"] >= 3)
]

all_default_loans_2021 = set(
    default_36m_perf_2021["LoanSequenceNumber"].unique()
)

print(
    "All 90+ DPD loans within 36 months:",
    len(all_default_loans_2021)
)

In [ ]:
eligible_default_loans_2021 = (
    all_default_loans_2021 &
    full_window_loans_2021
)

excluded_default_loans_2021 = (
    all_default_loans_2021 -
    full_window_loans_2021
)

print("Full-window loans:",
      len(full_window_loans_2021))

print("All 90+ DPD loans within 36M:",
      len(all_default_loans_2021))

print("Eligible 90+ DPD loans:",
      len(eligible_default_loans_2021))

print(
    "Defaults excluded because not full-window:",
    len(excluded_default_loans_2021)
)

In [ ]:
default_rate_2021 = (
    len(eligible_default_loans_2021) /
    len(full_window_loans_2021)
)

print(
    "Corrected 2021 36-month default rate:",
    default_rate_2021
)

print(
    "Corrected 2021 36-month default rate (%):",
    default_rate_2021 * 100
)

In [ ]:
target_2021 = pd.DataFrame({
    "LoanSequenceNumber": list(full_window_loans_2021)
})

target_2021["Default_36M"] = (
    target_2021["LoanSequenceNumber"]
    .isin(eligible_default_loans_2021)
    .astype(int)
)

print("Target dataset shape:", target_2021.shape)

print("\nTarget distribution:")
print(target_2021["Default_36M"].value_counts())

print("\nTarget proportions:")
print(
    target_2021["Default_36M"]
    .value_counts(normalize=True)
)

print("\nDefault rate:",
      target_2021["Default_36M"].mean())

In [ ]:
# ============================================================
# LOAD 2021 ORIGINATION FILE
# ============================================================

orig_2021 = pd.read_csv(
    os.path.join(
        extract_path_2021,
        "sample_orig_2021.txt"
    ),
    sep="|",
    header=None,
    low_memory=False
)

print("Origination shape:", orig_2021.shape)
print("Number of columns:", len(orig_2021.columns))

print("\nFirst row:")
for i, value in enumerate(orig_2021.iloc[0]):
    print(i, "=>", repr(value))

print(
    "\nUnique LoanSequenceNumber:",
    orig_2021[19].nunique()
)

In [ ]:
# ============================================================
# 2021 ORIGINATION COLUMN NAMES
# ============================================================

orig_columns_2021 = [
    "CreditScore",
    "FirstPaymentDate",
    "FirstTimeHomebuyerFlag",
    "MaturityDate",
    "MIPercent",
    "NumberOfUnits",
    "OccupancyStatus",
    "OriginalCLTV",
    "OriginalDTI",
    "OriginalUPB",
    "OriginalLTV",
    "OriginalInterestRate",
    "Channel",
    "PropertyState",
    "PropertyType",
    "AmortizationType",
    "PropertyState_2",
    "PropertyType_2",
    "PostalCode",
    "LoanSequenceNumber",
    "LoanPurpose",
    "OriginalLoanTerm",
    "NumberOfBorrowers",
    "SellerName",
    "ServicerName",
    "SuperConformingFlag",
    "PreReliefRefinanceLoanSequenceNumber",
    "SpecialEligibilityProgram",
    "ReliefRefinanceIndicator",
    "PropertyValuationMethod",
    "InterestOnlyIndicator"
]

print("Number of names:", len(orig_columns_2021))
print("Number of columns:", len(orig_2021.columns))

orig_2021.columns = orig_columns_2021

print("\nShape:", orig_2021.shape)

print("\nColumns:")
print(orig_2021.columns.tolist())

print(
    "\nUnique LoanSequenceNumber:",
    orig_2021["LoanSequenceNumber"].nunique()
)

In [ ]:
# ============================================================
# MERGE 2021 ORIGINATION + 36M TARGET
# ============================================================

model_2021 = orig_2021.merge(
    target_2021,
    on="LoanSequenceNumber",
    how="inner",
    validate="one_to_one"
)

print("Model dataset shape:", model_2021.shape)

print("\nTarget distribution:")
print(model_2021["Default_36M"].value_counts())

print("\nTarget proportions:")
print(
    model_2021["Default_36M"]
    .value_counts(normalize=True)
)

print("\nMissing target values:",
      model_2021["Default_36M"].isna().sum())

print("\nOriginal 2021 loans:", len(orig_2021))
print("Eligible modelling loans:", len(model_2021))
print(
    "Excluded from modelling:",
    len(orig_2021) - len(model_2021)
)

In [ ]:
# ============================================================
# 2021 MISSING-VALUE INSPECTION
# ============================================================

print("Starting shape:", model_2021.shape)

print("\nMissing values:")
print(
    model_2021.isna()
    .sum()
    .sort_values(ascending=False)
)

In [ ]:
# ============================================================
# 2021 SPECIAL MISSING-VALUE CODES
# ============================================================

print("CreditScore = 9999:",
      (model_2021["CreditScore"] == 9999).sum())

print("OriginalDTI = 999:",
      (model_2021["OriginalDTI"] == 999).sum())

print("OriginalLTV = 999:",
      (model_2021["OriginalLTV"] == 999).sum())

print("OriginalCLTV = 999:",
      (model_2021["OriginalCLTV"] == 999).sum())

print("OriginalInterestRate = 999:",
      (model_2021["OriginalInterestRate"] == 999).sum())

print("MIPercent missing:",
      model_2021["MIPercent"].isna().sum())

In [ ]:
# ============================================================
# CLEAN 2021 STRUCTURAL MISSINGNESS + SPECIAL CODES
# ============================================================

model_2021_clean = model_2021.copy()

# Drop unusable variables
drop_cols_2021 = [
    "SuperConformingFlag",
    "PreReliefRefinanceLoanSequenceNumber"
]

model_2021_clean = model_2021_clean.drop(
    columns=drop_cols_2021
)

# Convert Freddie Mac special missing-value codes to NaN
model_2021_clean.loc[
    model_2021_clean["CreditScore"] == 9999,
    "CreditScore"
] = np.nan

model_2021_clean.loc[
    model_2021_clean["OriginalDTI"] == 999,
    "OriginalDTI"
] = np.nan

model_2021_clean.loc[
    model_2021_clean["OriginalLTV"] == 999,
    "OriginalLTV"
] = np.nan

model_2021_clean.loc[
    model_2021_clean["OriginalCLTV"] == 999,
    "OriginalCLTV"
] = np.nan

model_2021_clean.loc[
    model_2021_clean["OriginalInterestRate"] == 999,
    "OriginalInterestRate"
] = np.nan

print("Shape after dropping unusable variables:",
      model_2021_clean.shape)

print("\nMissing values after converting special codes:")
print(
    model_2021_clean.isna()
    .sum()
    .sort_values(ascending=False)
)

In [ ]:
# ============================================================
# CHECK MISSINGNESS IN 2021
# ============================================================

missing_mask_2021 = model_2021_clean.isna().any(axis=1)

missing_rows_2021 = model_2021_clean.loc[
    missing_mask_2021
].copy()

print(
    "Number of rows with missing values:",
    len(missing_rows_2021)
)

print("\nMissing-value counts:")
print(
    model_2021_clean.isna()
    .sum()
    .sort_values(ascending=False)
)

print("\nDefault distribution among missing-value rows:")
print(
    missing_rows_2021["Default_36M"]
    .value_counts()
)

print(
    "\nDefault rate among missing-value rows:",
    missing_rows_2021["Default_36M"].mean()
)

print(
    "\nOverall default rate:",
    model_2021_clean["Default_36M"].mean()
)

In [ ]:
# ============================================================
# 2021: INSPECT SPECIAL-MISSING / MISSING ROWS
# ============================================================

print("Missing MIPercent rows:",
      model_2021_clean["MIPercent"].isna().sum())

print("Missing CreditScore rows:",
      model_2021_clean["CreditScore"].isna().sum())

print("\nDefault rate among MIPercent-missing rows:")
print(
    model_2021_clean.loc[
        model_2021_clean["MIPercent"].isna(),
        "Default_36M"
    ].mean()
)

print("\nDefault rate among CreditScore-missing rows:")
print(
    model_2021_clean.loc[
        model_2021_clean["CreditScore"].isna(),
        "Default_36M"
    ].mean()
)

print("\nOverall default rate:")
print(model_2021_clean["Default_36M"].mean())

In [ ]:
print(
    model_2021_clean["MIPercent"]
    .value_counts(dropna=False)
    .sort_index()
    .head(30)
)

In [ ]:
print(
    model_2021_clean.groupby(
        model_2021_clean["MIPercent"].isna()
    )["Default_36M"].agg(
        ["count", "sum", "mean"]
    )
)

In [ ]:
print("2021 raw column 4 examples:")
print(orig_2021["MIPercent"].head(20).tolist())

print("\n2021 raw column 8 examples:")
print(orig_2021["OriginalCLTV"].head(20).tolist())

print("\n2021 raw column 9 examples:")
print(orig_2021["OriginalDTI"].head(20).tolist())

print("\n2021 raw column 11 examples:")
print(orig_2021["OriginalLTV"].head(20).tolist())

In [ ]:
orig_columns_correct = [
    "CreditScore",                    # 0
    "FirstPaymentDate",               # 1
    "FirstTimeHomebuyerFlag",         # 2
    "MaturityDate",                   # 3
    "MIPercent",                      # 4
    "NumberOfUnits",                  # 5
    "OccupancyStatus",                # 6
    "OriginalCLTV",                   # 7
    "OriginalDTI",                    # 8
    "OriginalUPB",                    # 9
    "OriginalLTV",                    # 10
    "OriginalInterestRate",           # 11
    "Channel",                         # 12
    "PrepaymentPenaltyMortgageFlag",  # 13
    "PropertyState",                  # 14
    "PropertyType",                   # 15
    "PostalCode",                     # 16
    "LoanSequenceNumber",             # 17
    "LoanPurpose",                    # 18
    "OriginalLoanTerm",               # 19
    "NumberOfBorrowers",              # 20
    "SellerName",                     # 21
    "ServicerName",                   # 22
    "SuperConformingFlag",            # 23
    "PreReliefRefinanceLoanSequenceNumber", # 24
    "SpecialEligibilityProgram",      # 25
    "ReliefRefinanceIndicator",       # 26
    "PropertyValuationMethod",        # 27
    "InterestOnlyIndicator",          # 28
    "MI_cancellation_indicator",      # 29
    "Step_modification_indicator"     # 30
]

In [ ]:
orig_2021_check = orig_2021.copy()

orig_2021_check.columns = orig_columns_correct

print(orig_2021_check.iloc[0].to_dict())

In [ ]:
print("\nKey variable ranges:")

for col in [
    "MIPercent",
    "NumberOfUnits",
    "OriginalCLTV",
    "OriginalDTI",
    "OriginalUPB",
    "OriginalLTV",
    "OriginalInterestRate"
]:
    print(
        col,
        "min =", orig_2021_check[col].min(),
        "max =", orig_2021_check[col].max()
    )

In [ ]:
orig_columns_correct = [
    "CreditScore",                         # 0 / position 1
    "FirstPaymentDate",                    # 1 / position 2
    "FirstTimeHomebuyerFlag",              # 2 / position 3
    "MaturityDate",                        # 3 / position 4
    "MSA",                                 # 4 / position 5
    "MIPercent",                           # 5 / position 6
    "NumberOfUnits",                       # 6 / position 7
    "OccupancyStatus",                     # 7 / position 8
    "OriginalCLTV",                        # 8 / position 9
    "OriginalDTI",                         # 9 / position 10
    "OriginalUPB",                         # 10 / position 11
    "OriginalLTV",                         # 11 / position 12
    "OriginalInterestRate",                # 12 / position 13
    "Channel",                             # 13 / position 14
    "PPMFlag",                             # 14 / position 15
    "AmortizationType",                    # 15 / position 16
    "PropertyState",                       # 16 / position 17
    "PropertyType",                        # 17 / position 18
    "PostalCode",                          # 18 / position 19
    "LoanSequenceNumber",                  # 19 / position 20
    "LoanPurpose",                         # 20 / position 21
    "OriginalLoanTerm",                    # 21 / position 22
    "NumberOfBorrowers",                   # 22 / position 23
    "SellerName",                          # 23 / position 24
    "ServicerName",                        # 24 / position 25
    "SuperConformingFlag",                 # 25 / position 26
    "PreReliefRefinanceLoanSequenceNumber",# 26 / position 27
    "SpecialEligibilityProgram",           # 27 / position 28
    "ReliefRefinanceIndicator",            # 28 / position 29
    "PropertyValuationMethod",             # 29 / position 30
    "InterestOnlyIndicator"                # 30 / position 31
]

In [ ]:
# ============================================================
# REBUILD 2021 ORIGINATION MAPPING — CORRECT 31-COLUMN FORMAT
# ============================================================

orig_2021_correct = orig_2021.copy()

orig_2021_correct.columns = orig_columns_correct

print("Shape:", orig_2021_correct.shape)

print("\nFirst row:")
print(orig_2021_correct.iloc[0].to_dict())

print("\nUnique LoanSequenceNumber:",
      orig_2021_correct["LoanSequenceNumber"].nunique())

In [ ]:
# ============================================================
# CHECK KEY VARIABLES
# ============================================================

for col in [
    "MSA",
    "MIPercent",
    "NumberOfUnits",
    "OriginalCLTV",
    "OriginalDTI",
    "OriginalUPB",
    "OriginalLTV",
    "OriginalInterestRate",
    "OriginalLoanTerm",
    "NumberOfBorrowers"
]:
    print(
        col,
        "| min:", orig_2021_correct[col].min(),
        "| max:", orig_2021_correct[col].max(),
        "| missing:", orig_2021_correct[col].isna().sum()
    )

In [ ]:
# ============================================================
# 2021 CORRECTED ORIGINATION + 36M TARGET
# ============================================================

model_2021_correct = orig_2021_correct.merge(
    target_2021,
    on="LoanSequenceNumber",
    how="inner",
    validate="one_to_one"
)

print("Corrected 2021 model shape:", model_2021_correct.shape)

print("\nTarget distribution:")
print(model_2021_correct["Default_36M"].value_counts())

print("\nTarget proportions:")
print(
    model_2021_correct["Default_36M"]
    .value_counts(normalize=True)
)

print("\nMissing target values:",
      model_2021_correct["Default_36M"].isna().sum())

print("\nOriginal 2021 loans:", len(orig_2021_correct))
print("Eligible modelling loans:", len(model_2021_correct))
print(
    "Excluded from modelling:",
    len(orig_2021_correct) - len(model_2021_correct)
)

In [ ]:
print("\nMissing values:")
print(
    model_2021_correct.isna()
    .sum()
    .sort_values(ascending=False)
)

In [ ]:
# ============================================================
# 2021 CORRECTED CLEANING
# ============================================================

drop_cols_2021 = [
    "SuperConformingFlag",
    "PreReliefRefinanceLoanSequenceNumber",
    "MSA"
]

model_2021_final = model_2021_correct.drop(
    columns=drop_cols_2021
).copy()

print("Shape after dropping unusable columns:",
      model_2021_final.shape)

print("\nRemaining missing values:")
print(
    model_2021_final.isna()
    .sum()
    .sort_values(ascending=False)
)

In [ ]:
# ============================================================
# FINAL 2021 VARIABLE SANITY CHECK
# ============================================================

for col in [
    "CreditScore",
    "MIPercent",
    "NumberOfUnits",
    "OriginalCLTV",
    "OriginalDTI",
    "OriginalUPB",
    "OriginalLTV",
    "OriginalInterestRate",
    "OriginalLoanTerm",
    "NumberOfBorrowers"
]:
    print(
        col,
        "| min:", model_2021_final[col].min(),
        "| max:", model_2021_final[col].max(),
        "| missing:", model_2021_final[col].isna().sum()
    )

In [ ]:
# ============================================================
# CONVERT 2021 SPECIAL CREDIT SCORE CODE
# ============================================================

model_2021_final.loc[
    model_2021_final["CreditScore"] == 9999,
    "CreditScore"
] = np.nan

print(
    "CreditScore missing after conversion:",
    model_2021_final["CreditScore"].isna().sum()
)

In [ ]:
missing_cs_2021 = model_2021_final[
    model_2021_final["CreditScore"].isna()
]

print("Rows:", len(missing_cs_2021))

print("\nDefault distribution:")
print(
    missing_cs_2021["Default_36M"].value_counts()
)

print("\nDefault rate:")
print(
    missing_cs_2021["Default_36M"].mean()
)

In [ ]:
# Convert special DTI code
model_2021_final.loc[
    model_2021_final["OriginalDTI"] == 999,
    "OriginalDTI"
] = np.nan

print(
    "OriginalDTI missing after conversion:",
    model_2021_final["OriginalDTI"].isna().sum()
)

In [ ]:
print(
    model_2021_final[
        ["CreditScore", "OriginalDTI"]
    ].isna().sum()
)

In [ ]:
# ============================================================
# INSPECT 2021 SPECIAL-CODE ROWS
# ============================================================

missing_dti_2021 = model_2021_final[
    model_2021_final["OriginalDTI"].isna()
]

print("DTI-missing rows:", len(missing_dti_2021))

print("\nDTI-missing default distribution:")
print(
    missing_dti_2021["Default_36M"].value_counts()
)

print("\nDTI-missing default rate:")
print(
    missing_dti_2021["Default_36M"].mean()
)

print("\nDTI-missing rows:")
print(
    missing_dti_2021[
        [
            "LoanSequenceNumber",
            "Default_36M",
            "CreditScore",
            "OriginalDTI",
            "OriginalLTV",
            "OriginalCLTV",
            "OriginalUPB"
        ]
    ]
)

In [ ]:
# ============================================================
# COMBINED SPECIAL-MISSINGNESS CHECK
# ============================================================

special_missing_2021 = model_2021_final[
    model_2021_final[
        ["CreditScore", "OriginalDTI"]
    ].isna().any(axis=1)
]

print("Rows with CreditScore or DTI missing:",
      len(special_missing_2021))

print("\nDefault distribution:")
print(
    special_missing_2021["Default_36M"].value_counts()
)

print("\nDefault rate:")
print(
    special_missing_2021["Default_36M"].mean()
)

In [ ]:
# ============================================================
# FINAL SPECIAL-CODE HANDLING — 2021
# ============================================================

model_2021_final = model_2021_final.copy()

# Freddie Mac special missing-value codes
model_2021_final.loc[
    model_2021_final["CreditScore"] == 9999,
    "CreditScore"
] = np.nan

model_2021_final.loc[
    model_2021_final["OriginalDTI"] == 999,
    "OriginalDTI"
] = np.nan

print("CreditScore missing:",
      model_2021_final["CreditScore"].isna().sum())

print("DTI missing:",
      model_2021_final["OriginalDTI"].isna().sum())

print("\nFinal shape:",
      model_2021_final.shape)

print("\nTarget distribution:")
print(model_2021_final["Default_36M"].value_counts())

print("\nDefault rate:",
      model_2021_final["Default_36M"].mean())

In [ ]:
# ============================================================
# 2018 FREDDIE MAC SAMPLE — EXTRACT
# ============================================================

import os
import zipfile
import pandas as pd
import numpy as np

zip_path_2018 = r"./data/raw/sample_2018.zip"
extract_path_2018 = r"./data/raw/sample_2018"

os.makedirs(extract_path_2018, exist_ok=True)

with zipfile.ZipFile(zip_path_2018, "r") as z:
    z.extractall(extract_path_2018)

print("Extracted files:")
print(os.listdir(extract_path_2018))

In [ ]:
# ============================================================
# LOAD 2018 ORIGINATION + PERFORMANCE FILES
# ============================================================

orig_2018 = pd.read_csv(
    os.path.join(
        extract_path_2018,
        "sample_orig_2018.txt"
    ),
    sep="|",
    header=None,
    low_memory=False
)

perf_2018 = pd.read_csv(
    os.path.join(
        extract_path_2018,
        "sample_perf_2018.txt"
    ),
    sep="|",
    header=None,
    low_memory=False
)

print("Origination shape:", orig_2018.shape)
print("Performance shape:", perf_2018.shape)

print("\nOrigination columns:", len(orig_2018.columns))
print("Performance columns:", len(perf_2018.columns))

In [ ]:
# ============================================================
# RAW 2018 ORIGINATION STRUCTURE
# ============================================================

print("First row of 2018 origination file:\n")

for i, value in enumerate(orig_2018.iloc[0]):
    print(i, "=>", repr(value))

In [ ]:
# ============================================================
# RAW 2018 PERFORMANCE STRUCTURE
# ============================================================

print("\nFirst row of 2018 performance file:\n")

for i, value in enumerate(perf_2018.iloc[0]):
    print(i, "=>", repr(value))

In [ ]:
# ============================================================
# FREDDIE MAC PROJECT — 2018 PD DATASET
# Project: End-to-End Mortgage Credit Risk Modelling
# Phase 2: Probability of Default (PD)
# Target: 90+ DPD within 36 months
# ============================================================

import os
import zipfile
import pandas as pd
import numpy as np

# ============================================================
# 1. FILE PATH
# ============================================================

zip_path_2018 = r"./data/raw/sample_2018.zip"

extract_path_2018 = r"./data/raw/sample_2018"

os.makedirs(extract_path_2018, exist_ok=True)

with zipfile.ZipFile(zip_path_2018, "r") as z:
    z.extractall(extract_path_2018)

print("Extracted files:")
print(os.listdir(extract_path_2018))


# ============================================================
# 2. LOAD ORIGINATION + PERFORMANCE FILES
# ============================================================

orig_file_2018 = os.path.join(
    extract_path_2018,
    "sample_orig_2018.txt"
)

perf_file_2018 = os.path.join(
    extract_path_2018,
    "sample_perf_2018.txt"
)

orig_2018 = pd.read_csv(
    orig_file_2018,
    sep="|",
    header=None,
    low_memory=False
)

perf_2018 = pd.read_csv(
    perf_file_2018,
    sep="|",
    header=None,
    low_memory=False
)

print("Origination shape:", orig_2018.shape)
print("Performance shape:", perf_2018.shape)


# ============================================================
# 3. PERFORMANCE CHECK
# ============================================================

print("\nMaximum LoanAge:", perf_2018[4].max())
print("Minimum LoanAge:", perf_2018[4].min())

print("\nPerformance columns:", len(perf_2018.columns))

print("\nColumn 3 value counts:")
print(
    perf_2018[3]
    .value_counts(dropna=False)
    .head(30)
)


# ============================================================
# 4. IDENTIFY LOANS WITH COMPLETE 36-MONTH WINDOW
#
# LoanAge 35 means months 0–35 are available = 36 months.
# ============================================================

full_window_loans_2018 = set(
    perf_2018.loc[
        perf_2018[4] >= 35,
        0
    ].unique()
)

print(
    "\nFull-window 2018 loans:",
    len(full_window_loans_2018)
)


# ============================================================
# 5. CONVERT DELINQUENCY STATUS TO NUMERIC
#
# Column 3 contains delinquency status.
# "00" = current
# "01", "02", ... = delinquency status
# RA is a non-numeric status and therefore becomes NaN.
# ============================================================

perf_2018["DelinqNumeric"] = pd.to_numeric(
    perf_2018[3],
    errors="coerce"
)

print("\nDelinqNumeric value counts:")
print(
    perf_2018["DelinqNumeric"]
    .value_counts(dropna=False)
    .head(30)
)


# ============================================================
# 6. IDENTIFY 90+ DPD
#
# 90+ DPD = delinquency status 03 or greater.
# Restrict to LoanAge 0–35.
# ============================================================

default_36m_perf_2018 = perf_2018.loc[
    (perf_2018[4] <= 35) &
    (perf_2018["DelinqNumeric"] >= 3)
].copy()

all_default_loans_2018 = set(
    default_36m_perf_2018[0].unique()
)

print(
    "\nAll loans with 90+ DPD within 36 months:",
    len(all_default_loans_2018)
)


# ============================================================
# 7. KEEP ONLY DEFAULTS WITH COMPLETE 36-MONTH OBSERVATION
# ============================================================

eligible_default_loans_2018 = (
    all_default_loans_2018 &
    full_window_loans_2018
)

excluded_default_loans_2018 = (
    all_default_loans_2018 -
    full_window_loans_2018
)

print("\nFull-window loans:",
      len(full_window_loans_2018))

print("All 90+ DPD loans within 36M:",
      len(all_default_loans_2018))

print("Eligible 90+ DPD loans:",
      len(eligible_default_loans_2018))

print("Defaults excluded because not full-window:",
      len(excluded_default_loans_2018))


# ============================================================
# 8. CALCULATE 2018 DEFAULT RATE
# ============================================================

default_rate_2018 = (
    len(eligible_default_loans_2018) /
    len(full_window_loans_2018)
)

print(
    "\nCorrected 2018 36-month default rate:",
    default_rate_2018
)

print(
    "Corrected 2018 36-month default rate (%):",
    default_rate_2018 * 100
)


# ============================================================
# 9. CREATE PD TARGET
# ============================================================

target_2018 = pd.DataFrame({
    "LoanSequenceNumber":
        list(full_window_loans_2018)
})

target_2018["Default_36M"] = (
    target_2018["LoanSequenceNumber"]
    .isin(eligible_default_loans_2018)
    .astype(int)
)

print("\nTarget dataset shape:",
      target_2018.shape)

print("\nTarget distribution:")
print(
    target_2018["Default_36M"]
    .value_counts()
)

print("\nTarget proportions:")
print(
    target_2018["Default_36M"]
    .value_counts(normalize=True)
)

print(
    "\nDefault rate:",
    target_2018["Default_36M"].mean()
)


# ============================================================
# 10. CHECK ORIGINATION FILE
# ============================================================

print("\nOrigination shape:",
      orig_2018.shape)

print("Number of columns:",
      len(orig_2018.columns))

print("\nFirst row:")
for i, value in orig_2018.iloc[0].items():
    print(i, "=>", repr(value))


# ============================================================
# 11. IMPORTANT:
# 2018 MAY HAVE A DIFFERENT FORMAT.
#
# DO NOT blindly assign the 2020/2021 column list.
# First inspect the number of columns.
# ============================================================

print(
    "\n2018 origination column count:",
    len(orig_2018.columns)
)

print("\n2018 first-row values:")
print(orig_2018.iloc[0].tolist())

In [ ]:
# ============================================================
# 2018 ORIGINATION COLUMN NAMES
# ============================================================

orig_columns_2018 = [
    "CreditScore",
    "FirstPaymentDate",
    "FirstTimeHomebuyerFlag",
    "MaturityDate",
    "MIPercent",
    "NumberOfUnits",
    "OccupancyStatus",
    "OriginalCLTV",
    "OriginalDTI",
    "OriginalUPB",
    "OriginalLTV",
    "OriginalInterestRate",
    "Channel",
    "PPMFlag",
    "AmortizationType",
    "PropertyState",
    "PropertyType",
    "PostalCode",
    "LoanSequenceNumber",
    "LoanPurpose",
    "OriginalLoanTerm",
    "NumberOfBorrowers",
    "SellerName",
    "ServicerName",
    "SuperConformingFlag",
    "PreReliefRefinanceLoanSequenceNumber",
    "SpecialEligibilityProgram",
    "ReliefRefinanceIndicator",
    "PropertyValuationMethod",
    "InterestOnlyIndicator",
    "MSA"
]

print("Number of names:", len(orig_columns_2018))
print("Number of columns:", len(orig_2018.columns))

orig_2018.columns = orig_columns_2018

print("\nShape:", orig_2018.shape)

print("\nColumns:")
print(orig_2018.columns.tolist())

print(
    "\nUnique LoanSequenceNumber:",
    orig_2018["LoanSequenceNumber"].nunique()
)

In [ ]:
# ============================================================
# CHECK 2018 MERGE KEY
# ============================================================

print("orig_2018 LoanSequenceNumber:")
print(orig_2018["LoanSequenceNumber"].head())
print("dtype:", orig_2018["LoanSequenceNumber"].dtype)

print("\ntarget_2018 LoanSequenceNumber:")
print(target_2018["LoanSequenceNumber"].head())
print("dtype:", target_2018["LoanSequenceNumber"].dtype)

In [ ]:
orig_columns_2018 = [
    "CreditScore",                         # 0
    "FirstPaymentDate",                    # 1
    "FirstTimeHomebuyerFlag",              # 2
    "MaturityDate",                        # 3
    "MIPercent",                           # 4
    "NumberOfUnits",                       # 5
    "OccupancyStatus",                     # 6
    "OriginalCLTV",                        # 7
    "OriginalDTI",                         # 8
    "OriginalUPB",                         # 9
    "OriginalLTV",                         # 10
    "OriginalInterestRate",                # 11
    "Channel",                             # 12
    "PPMFlag",                             # 13
    "AmortizationType",                    # 14
    "PropertyState",                       # 15
    "PropertyType",                        # 16
    "PostalCode",                          # 17
    "LoanSequenceNumber",                  # 18
    "LoanPurpose",                         # 19
    "OriginalLoanTerm",                    # 20
    "NumberOfBorrowers",                   # 21
    "SellerName",                          # 22
    "ServicerName",                        # 23
    "SuperConformingFlag",                 # 24
    "PreReliefRefinanceLoanSequenceNumber",# 25
    "SpecialEligibilityProgram",           # 26
    "ReliefRefinanceIndicator",            # 27
    "PropertyValuationMethod",             # 28
    "InterestOnlyIndicator",               # 29
    "MSA"                                  # 30
]

In [ ]:
# ============================================================
# REBUILD 2018 COLUMN NAMES USING THE OBSERVED FILE STRUCTURE
# ============================================================

orig_columns_2018 = [
    "CreditScore",
    "FirstPaymentDate",
    "FirstTimeHomebuyerFlag",
    "MaturityDate",
    "MIPercent",
    "NumberOfUnits",
    "OccupancyStatus",
    "OriginalCLTV",
    "OriginalDTI",
    "OriginalUPB",
    "OriginalLTV",
    "OriginalInterestRate",
    "Channel",
    "PPMFlag",
    "AmortizationType",
    "PropertyState",
    "PropertyType",
    "PostalCode",
    "LoanSequenceNumber",
    "LoanPurpose",
    "OriginalLoanTerm",
    "NumberOfBorrowers",
    "SellerName",
    "ServicerName",
    "SuperConformingFlag",
    "PreReliefRefinanceLoanSequenceNumber",
    "SpecialEligibilityProgram",
    "ReliefRefinanceIndicator",
    "PropertyValuationMethod",
    "InterestOnlyIndicator",
    "MSA"
]

print("Names:", len(orig_columns_2018))
print("Columns:", len(orig_2018.columns))

orig_2018.columns = orig_columns_2018

print("\nLoanSequenceNumber sample:")
print(orig_2018["LoanSequenceNumber"].head())

print(
    "Origination key dtype:",
    orig_2018["LoanSequenceNumber"].dtype
)

print(
    "Target key dtype:",
    target_2018["LoanSequenceNumber"].dtype
)

In [ ]:
# ============================================================
# STANDARDIZE LOAN IDENTIFIER
# ============================================================

orig_2018["LoanSequenceNumber"] = (
    orig_2018["LoanSequenceNumber"]
    .astype(str)
    .str.strip()
)

target_2018["LoanSequenceNumber"] = (
    target_2018["LoanSequenceNumber"]
    .astype(str)
    .str.strip()
)

print(
    "Origination unique IDs:",
    orig_2018["LoanSequenceNumber"].nunique()
)

print(
    "Target unique IDs:",
    target_2018["LoanSequenceNumber"].nunique()
)

print(
    "\nExample origination ID:",
    orig_2018["LoanSequenceNumber"].iloc[0]
)

print(
    "Example target ID:",
    target_2018["LoanSequenceNumber"].iloc[0]
)

In [ ]:
# ============================================================
# MERGE 2018 ORIGINATION + TARGET
# ============================================================

model_2018 = orig_2018.merge(
    target_2018,
    on="LoanSequenceNumber",
    how="inner",
    validate="one_to_one"
)

print("Model dataset shape:", model_2018.shape)

print("\nTarget distribution:")
print(
    model_2018["Default_36M"].value_counts()
)

print("\nTarget proportions:")
print(
    model_2018["Default_36M"]
    .value_counts(normalize=True)
)

print(
    "\nOriginal 2018 loans:",
    len(orig_2018)
)

print(
    "Eligible modelling loans:",
    len(model_2018)
)

print(
    "Excluded from modelling:",
    len(orig_2018) - len(model_2018)
)

In [ ]:
# ============================================================
# 2018 — REBUILD ORIGINATION DATAFRAME CLEANLY
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. READ RAW 2018 ORIGINATION FILE
# ------------------------------------------------------------

orig_2018 = pd.read_csv(
    r"./data/raw/sample_orig_2018.txt",
    sep="|",
    header=None
)

print("Raw origination shape:", orig_2018.shape)

# ------------------------------------------------------------
# 2. CHECK RAW COLUMN COUNT
# ------------------------------------------------------------

print("Raw number of columns:", len(orig_2018.columns))

# ------------------------------------------------------------
# 3. OFFICIAL 31-COLUMN ORIGINATION STRUCTURE
# ------------------------------------------------------------

orig_columns_2018 = [
    "CreditScore",
    "FirstPaymentDate",
    "FirstTimeHomebuyerFlag",
    "MaturityDate",
    "MIPercent",
    "NumberOfUnits",
    "OccupancyStatus",
    "OriginalCLTV",
    "OriginalDTI",
    "OriginalUPB",
    "OriginalLTV",
    "OriginalInterestRate",
    "Channel",
    "PropertyState",
    "PropertyType",
    "AmortizationType",
    "PropertyState_2",
    "PropertyType_2",
    "PostalCode",
    "LoanSequenceNumber",
    "LoanPurpose",
    "OriginalLoanTerm",
    "NumberOfBorrowers",
    "SellerName",
    "ServicerName",
    "SuperConformingFlag",
    "PreReliefRefinanceLoanSequenceNumber",
    "SpecialEligibilityProgram",
    "ReliefRefinanceIndicator",
    "PropertyValuationMethod",
    "InterestOnlyIndicator"
]

assert len(orig_columns_2018) == 31
assert len(orig_2018.columns) == 31

orig_2018.columns = orig_columns_2018

# ------------------------------------------------------------
# 4. FORCE LOAN ID TO STRING
# ------------------------------------------------------------

orig_2018["LoanSequenceNumber"] = (
    orig_2018["LoanSequenceNumber"]
    .astype(str)
    .str.strip()
)

# ------------------------------------------------------------
# 5. CHECK LOAN ID
# ------------------------------------------------------------

print("\nOrigination shape:", orig_2018.shape)

print(
    "Unique LoanSequenceNumber:",
    orig_2018["LoanSequenceNumber"].nunique()
)

print(
    "Duplicate LoanSequenceNumber:",
    orig_2018["LoanSequenceNumber"].duplicated().sum()
)

print(
    "\nFirst 5 LoanSequenceNumbers:"
)

print(
    orig_2018["LoanSequenceNumber"].head()
)

print(
    "\nFirst row:"
)

print(orig_2018.iloc[0].to_dict())

In [ ]:
import os

print(os.listdir(r"./data/raw"))

In [ ]:
import zipfile
import os
import pandas as pd
import numpy as np

# ============================================================
# FIND 2018 ZIP
# ============================================================

downloads = r"./data/raw"

files = os.listdir(downloads)

print("2018-related files:")
for f in files:
    if "2018" in f.lower():
        print(f)

# Find the 2018 ZIP
zip_candidates = [
    f for f in files
    if "2018" in f.lower() and f.lower().endswith(".zip")
]

print("\nZIP candidates:", zip_candidates)

assert len(zip_candidates) >= 1, "2018 ZIP file not found in Downloads."

zip_path = os.path.join(downloads, zip_candidates[0])

print("\nUsing:", zip_path)

# ============================================================
# EXTRACT
# ============================================================

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(downloads)
    extracted = z.namelist()

print("\nExtracted files:")
print(extracted)

# ============================================================
# FIND ORIGINATION FILE
# ============================================================

orig_candidates = [
    f for f in extracted
    if "orig" in f.lower() and "2018" in f.lower()
]

print("\nOrigination candidates:")
print(orig_candidates)

assert len(orig_candidates) >= 1, "2018 origination file not found."

orig_path = os.path.join(downloads, orig_candidates[0])

print("\nReading:", orig_path)

# ============================================================
# READ RAW ORIGINATION
# ============================================================

orig_2018 = pd.read_csv(
    orig_path,
    sep="|",
    header=None
)

print("\nRaw origination shape:", orig_2018.shape)
print("Raw number of columns:", len(orig_2018.columns))

In [ ]:
# ============================================================
# FREDDIE MAC PROJECT — 2018 PD DATASET
# STEP 1: REBUILD 2018 ORIGINATION CORRECTLY
# ============================================================

import pandas as pd
import numpy as np
import zipfile
import os

# ------------------------------------------------------------
# 1. Locate ZIP
# ------------------------------------------------------------

zip_path = r"./data/raw/sample_2018.zip"

with zipfile.ZipFile(zip_path, "r") as z:
    files_2018 = z.namelist()

print("Files in ZIP:")
print(files_2018)

# ------------------------------------------------------------
# 2. Extract origination file
# ------------------------------------------------------------

orig_file_2018 = [
    f for f in files_2018
    if "sample_orig_2018" in f.lower()
][0]

with zipfile.ZipFile(zip_path, "r") as z:
    z.extract(orig_file_2018, r"./data/raw")

orig_path_2018 = os.path.join(
    r"./data/raw",
    orig_file_2018
)

print("\nReading:")
print(orig_path_2018)

# ------------------------------------------------------------
# 3. Read raw origination file
# ------------------------------------------------------------

orig_2018_raw = pd.read_csv(
    orig_path_2018,
    sep="|",
    header=None
)

print("\nRaw shape:", orig_2018_raw.shape)
print("Raw columns:", len(orig_2018_raw.columns))

# ------------------------------------------------------------
# 4. IMPORTANT:
# 2018 sample has 31 fields.
# Use the 31-field Freddie Mac origination structure.
# ------------------------------------------------------------

orig_columns_2018 = [
    "CreditScore",
    "FirstPaymentDate",
    "FirstTimeHomebuyerFlag",
    "MaturityDate",
    "MIPercent",
    "NumberOfUnits",
    "OccupancyStatus",
    "OriginalCLTV",
    "OriginalDTI",
    "OriginalUPB",
    "OriginalLTV",
    "OriginalInterestRate",
    "Channel",
    "PPMFlag",
    "AmortizationType",
    "PropertyState",
    "PropertyType",
    "PostalCode",
    "LoanSequenceNumber",
    "LoanPurpose",
    "OriginalLoanTerm",
    "NumberOfBorrowers",
    "SellerName",
    "ServicerName",
    "SuperConformingFlag",
    "PreReliefRefinanceLoanSequenceNumber",
    "SpecialEligibilityProgram",
    "ReliefRefinanceIndicator",
    "PropertyValuationMethod",
    "InterestOnlyIndicator",
    "UnusedField"
]

print("\nNumber of column names:",
      len(orig_columns_2018))

# ------------------------------------------------------------
# 5. Assign names
# ------------------------------------------------------------

orig_2018 = orig_2018_raw.copy()
orig_2018.columns = orig_columns_2018

print("\nOrigination shape:", orig_2018.shape)

# ------------------------------------------------------------
# 6. Inspect first row
# ------------------------------------------------------------

print("\nFirst row:")
print(orig_2018.iloc[0].to_dict())

# ------------------------------------------------------------
# 7. Check LoanSequenceNumber
# ------------------------------------------------------------

print(
    "\nUnique LoanSequenceNumber:",
    orig_2018["LoanSequenceNumber"].nunique()
)

print(
    "Missing LoanSequenceNumber:",
    orig_2018["LoanSequenceNumber"].isna().sum()
)

print(
    "LoanSequenceNumber dtype:",
    orig_2018["LoanSequenceNumber"].dtype
)

In [ ]:
# ============================================================
# FREDDIE MAC — CORRECT 2018 ORIGINATION MAPPING
# ============================================================

orig_columns_2018_correct = [
    "CreditScore",
    "FirstPaymentDate",
    "FirstTimeHomebuyerFlag",
    "MaturityDate",
    "MSA",
    "MIPercent",
    "NumberOfUnits",
    "OccupancyStatus",
    "OriginalCLTV",
    "OriginalDTI",
    "OriginalUPB",
    "OriginalLTV",
    "OriginalInterestRate",
    "Channel",
    "PPMFlag",
    "AmortizationType",
    "PropertyState",
    "PropertyType",
    "PostalCode",
    "LoanSequenceNumber",
    "LoanPurpose",
    "OriginalLoanTerm",
    "NumberOfBorrowers",
    "SellerName",
    "ServicerName",
    "SuperConformingFlag",
    "PreReliefRefinanceLoanSequenceNumber",
    "SpecialEligibilityProgram",
    "ReliefRefinanceIndicator",
    "PropertyValuationMethod",
    "InterestOnlyIndicator"
]

# Start from the RAW dataframe, not the incorrectly renamed one
orig_2018_correct = orig_2018_raw.copy()

orig_2018_correct.columns = orig_columns_2018_correct

print("Shape:", orig_2018_correct.shape)

print("\nFirst row:")
print(orig_2018_correct.iloc[0].to_dict())

print(
    "\nUnique LoanSequenceNumber:",
    orig_2018_correct["LoanSequenceNumber"].nunique()
)

print(
    "Missing LoanSequenceNumber:",
    orig_2018_correct["LoanSequenceNumber"].isna().sum()
)

print(
    "LoanSequenceNumber dtype:",
    orig_2018_correct["LoanSequenceNumber"].dtype
)

In [ ]:
# ============================================================
# 2018 ORIGINATION + 36M TARGET
# ============================================================

model_2018 = orig_2018_correct.merge(
    target_2018,
    on="LoanSequenceNumber",
    how="inner",
    validate="one_to_one"
)

print("Model dataset shape:", model_2018.shape)

print("\nTarget distribution:")
print(model_2018["Default_36M"].value_counts())

print("\nTarget proportions:")
print(
    model_2018["Default_36M"]
    .value_counts(normalize=True)
)

print(
    "\nMissing target values:",
    model_2018["Default_36M"].isna().sum()
)

print("\nOriginal 2018 loans:", len(orig_2018_correct))
print("Eligible modelling loans:", len(model_2018))

print(
    "Excluded from modelling:",
    len(orig_2018_correct) - len(model_2018)
)

In [ ]:
# ============================================================
# 2018 — FINAL MISSING VALUE & SPECIAL CODE CHECK
# ============================================================

model_2018_clean = model_2018.copy()

print("Starting shape:", model_2018_clean.shape)

print("\nMissing values:")
print(
    model_2018_clean.isna()
    .sum()
    .sort_values(ascending=False)
)

In [ ]:
# ============================================================
# 2018 — CHECK SPECIAL MISSING-VALUE CODES
# ============================================================

print("CreditScore = 9999:",
      (model_2018_clean["CreditScore"] == 9999).sum())

print("OriginalDTI = 999:",
      (model_2018_clean["OriginalDTI"] == 999).sum())

print("OriginalLTV = 999:",
      (model_2018_clean["OriginalLTV"] == 999).sum())

print("OriginalCLTV = 999:",
      (model_2018_clean["OriginalCLTV"] == 999).sum())

print("OriginalInterestRate = 999:",
      (model_2018_clean["OriginalInterestRate"] == 999).sum())

print("MIPercent missing:",
      model_2018_clean["MIPercent"].isna().sum())

In [ ]:
# ============================================================
# 2018 — CONVERT SPECIAL MISSING-VALUE CODES
# ============================================================

model_2018_clean = model_2018.copy()

# Freddie Mac special codes
model_2018_clean.loc[
    model_2018_clean["CreditScore"] == 9999,
    "CreditScore"
] = np.nan

model_2018_clean.loc[
    model_2018_clean["OriginalDTI"] == 999,
    "OriginalDTI"
] = np.nan

print("CreditScore missing after conversion:",
      model_2018_clean["CreditScore"].isna().sum())

print("DTI missing after conversion:",
      model_2018_clean["OriginalDTI"].isna().sum())

print("\nDefault distribution among CreditScore-missing rows:")
print(
    model_2018_clean.loc[
        model_2018_clean["CreditScore"].isna(),
        "Default_36M"
    ].value_counts()
)

print("\nDefault distribution among DTI-missing rows:")
print(
    model_2018_clean.loc[
        model_2018_clean["OriginalDTI"].isna(),
        "Default_36M"
    ].value_counts()
)

In [ ]:
# ============================================================
# 2018 — DROP VARIABLES THAT ARE UNUSABLE
# ============================================================

drop_cols_2018 = [
    "MSA",
    "SuperConformingFlag",
    "PreReliefRefinanceLoanSequenceNumber"
]

model_2018_final = model_2018_clean.drop(
    columns=drop_cols_2018
)

print("Final 2018 shape:", model_2018_final.shape)

print("\nRemaining missing values:")
print(
    model_2018_final.isna()
    .sum()
    .sort_values(ascending=False)
)

In [ ]:
# ============================================================
# FREDDIE MAC PROJECT — 2019
# STEP 1: LOCATE AND EXTRACT SAMPLE
# ============================================================

import os
import zipfile
import pandas as pd
import numpy as np

downloads = r"./data/raw"

# Find 2019 ZIP
zip_candidates_2019 = [
    f for f in os.listdir(downloads)
    if "2019" in f.lower()
    and f.lower().endswith(".zip")
]

print("2019 ZIP candidates:")
print(zip_candidates_2019)

In [ ]:
# ============================================================
# FREDDIE MAC PROJECT — 2019
# STEP 1: EXTRACT + INITIAL INSPECTION
# ============================================================

import zipfile
import os
import pandas as pd
import numpy as np

zip_path_2019 = r"./data/raw/sample_2019.zip"
extract_dir = r"./data/raw"

with zipfile.ZipFile(zip_path_2019, "r") as z:
    files_2019 = z.namelist()
    print("Files in ZIP:")
    print(files_2019)

    z.extractall(extract_dir)

# ------------------------------------------------------------
# Locate files
# ------------------------------------------------------------

orig_path_2019 = os.path.join(
    extract_dir,
    "sample_orig_2019.txt"
)

perf_path_2019 = os.path.join(
    extract_dir,
    "sample_perf_2019.txt"
)

# ------------------------------------------------------------
# Read raw files
# ------------------------------------------------------------

orig_2019_raw = pd.read_csv(
    orig_path_2019,
    sep="|",
    header=None
)

perf_2019_raw = pd.read_csv(
    perf_path_2019,
    sep="|",
    header=None
)

print("\nOrigination shape:", orig_2019_raw.shape)
print("Performance shape:", perf_2019_raw.shape)

print("\nOrigination columns:",
      len(orig_2019_raw.columns))

print("Performance columns:",
      len(perf_2019_raw.columns))

print("\n2019 origination first row:")
print(orig_2019_raw.iloc[0].tolist())

print("\n2019 performance first row:")
print(perf_2019_raw.iloc[0].tolist())


In [ ]:
# ============================================================
# 2019 — CONSTRUCT 36-MONTH DEFAULT TARGET
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Identify the performance columns
# ------------------------------------------------------------

print("Performance shape:", perf_2019_raw.shape)
print("Performance columns:", perf_2019_raw.shape[1])

# LoanSequenceNumber is column 0
# Monthly reporting/performance date is column 1
# Delinquency status is column 3

perf_2019 = perf_2019_raw.copy()

# ------------------------------------------------------------
# 2. Rename only the fields we need
# ------------------------------------------------------------

perf_2019 = perf_2019.rename(columns={
    perf_2019.columns[0]: "LoanSequenceNumber",
    perf_2019.columns[1]: "MonthlyReportingPeriod",
    perf_2019.columns[3]: "DelinquencyStatus"
})

# Make loan ID consistent with origination
perf_2019["LoanSequenceNumber"] = (
    perf_2019["LoanSequenceNumber"]
    .astype(str)
    .str.strip()
)

orig_2019["LoanSequenceNumber"] = (
    orig_2019["LoanSequenceNumber"]
    .astype(str)
    .str.strip()
)

# ------------------------------------------------------------
# 3. Convert delinquency status to numeric
# ------------------------------------------------------------

perf_2019["DelinqNumeric"] = pd.to_numeric(
    perf_2019["DelinquencyStatus"],
    errors="coerce"
)

print("\nDelinquency values:")
print(
    perf_2019["DelinqNumeric"]
    .value_counts(dropna=False)
    .sort_index()
)

# ------------------------------------------------------------
# 4. Calculate maximum loan age observed
# ------------------------------------------------------------

# Loan age is column 2 in the Freddie Mac performance file
perf_2019["LoanAge"] = pd.to_numeric(
    perf_2019_raw.iloc[:, 2],
    errors="coerce"
)

print("\nLoanAge range:")
print("Minimum:", perf_2019["LoanAge"].min())
print("Maximum:", perf_2019["LoanAge"].max())

# ------------------------------------------------------------
# 5. Identify loans with a complete 36-month window
# ------------------------------------------------------------

loan_age_summary_2019 = (
    perf_2019
    .groupby("LoanSequenceNumber")["LoanAge"]
    .max()
)

full_window_loans_2019 = loan_age_summary_2019[
    loan_age_summary_2019 >= 35
].index

print(
    "\nFull-window 2019 loans:",
    len(full_window_loans_2019)
)

# ------------------------------------------------------------
# 6. Restrict performance data to first 36 months
# ------------------------------------------------------------

perf_2019_36m = perf_2019[
    perf_2019["LoanAge"].between(0, 35)
].copy()

# ------------------------------------------------------------
# 7. Identify 90+ DPD within 36 months
# ------------------------------------------------------------

default_90dpd_2019 = (
    perf_2019_36m
    .groupby("LoanSequenceNumber")["DelinqNumeric"]
    .max()
)

all_90dpd_2019 = default_90dpd_2019[
    default_90dpd_2019 >= 3
].index

print(
    "All loans with 90+ DPD within 36 months:",
    len(all_90dpd_2019)
)

# ------------------------------------------------------------
# 8. Keep only defaults from complete-window loans
# ------------------------------------------------------------

eligible_defaults_2019 = (
    all_90dpd_2019
    .intersection(full_window_loans_2019)
)

print(
    "Eligible 90+ DPD loans:",
    len(eligible_defaults_2019)
)

print(
    "Defaults excluded because not full-window:",
    len(all_90dpd_2019) - len(eligible_defaults_2019)
)

# ------------------------------------------------------------
# 9. Construct target
# ------------------------------------------------------------

target_2019 = pd.DataFrame({
    "LoanSequenceNumber": full_window_loans_2019
})

target_2019["Default_36M"] = (
    target_2019["LoanSequenceNumber"]
    .isin(eligible_defaults_2019)
    .astype(int)
)

# ------------------------------------------------------------
# 10. Check target
# ------------------------------------------------------------

print("\nTarget dataset shape:", target_2019.shape)

print("\nTarget distribution:")
print(
    target_2019["Default_36M"]
    .value_counts()
)

print("\nTarget proportions:")
print(
    target_2019["Default_36M"]
    .value_counts(normalize=True)
)

print(
    "\n2019 36-month default rate:",
    target_2019["Default_36M"].mean()
)

print(
    "\n2019 36-month default rate (%):",
    target_2019["Default_36M"].mean() * 100
)

In [ ]:
# ============================================================
# 2019 — CORRECT LOAN AGE AND 36-MONTH TARGET
# ============================================================

perf_2019 = perf_2019_raw.copy()

# ------------------------------------------------------------
# 1. Correct column mapping
# ------------------------------------------------------------

perf_2019 = perf_2019.rename(columns={
    perf_2019.columns[0]: "LoanSequenceNumber",
    perf_2019.columns[1]: "MonthlyReportingPeriod",
    perf_2019.columns[2]: "CurrentActualUPB",
    perf_2019.columns[3]: "DelinquencyStatus",
    perf_2019.columns[4]: "LoanAge"
})

# ------------------------------------------------------------
# 2. Standardize LoanSequenceNumber
# ------------------------------------------------------------

perf_2019["LoanSequenceNumber"] = (
    perf_2019["LoanSequenceNumber"]
    .astype(str)
    .str.strip()
)

orig_2019["LoanSequenceNumber"] = (
    orig_2019["LoanSequenceNumber"]
    .astype(str)
    .str.strip()
)

# ------------------------------------------------------------
# 3. Convert required fields
# ------------------------------------------------------------

perf_2019["DelinqNumeric"] = pd.to_numeric(
    perf_2019["DelinquencyStatus"],
    errors="coerce"
)

perf_2019["LoanAge"] = pd.to_numeric(
    perf_2019["LoanAge"],
    errors="coerce"
)

# ------------------------------------------------------------
# 4. Sanity check LoanAge
# ------------------------------------------------------------

print("Correct LoanAge range:")
print("Minimum:", perf_2019["LoanAge"].min())
print("Maximum:", perf_2019["LoanAge"].max())

print("\nLoanAge value counts:")
print(
    perf_2019["LoanAge"]
    .value_counts()
    .sort_index()
    .head(40)
)

# ------------------------------------------------------------
# 5. Full 36-month window
# ------------------------------------------------------------

loan_age_summary_2019 = (
    perf_2019
    .groupby("LoanSequenceNumber")["LoanAge"]
    .max()
)

full_window_loans_2019 = loan_age_summary_2019[
    loan_age_summary_2019 >= 35
].index

print(
    "\nFull-window 2019 loans:",
    len(full_window_loans_2019)
)

# ------------------------------------------------------------
# 6. Restrict to first 36 months
# ------------------------------------------------------------

perf_2019_36m = perf_2019[
    perf_2019["LoanAge"].between(0, 35)
].copy()

print(
    "Performance observations in 36M window:",
    len(perf_2019_36m)
)

# ------------------------------------------------------------
# 7. Identify 90+ DPD
# ------------------------------------------------------------

default_90dpd_2019 = (
    perf_2019_36m
    .groupby("LoanSequenceNumber")["DelinqNumeric"]
    .max()
)

all_90dpd_2019 = default_90dpd_2019[
    default_90dpd_2019 >= 3
].index

print(
    "\nAll loans with 90+ DPD within 36 months:",
    len(all_90dpd_2019)
)

# ------------------------------------------------------------
# 8. Keep only complete-window defaults
# ------------------------------------------------------------

eligible_defaults_2019 = (
    all_90dpd_2019
    .intersection(full_window_loans_2019)
)

print(
    "Eligible 90+ DPD loans:",
    len(eligible_defaults_2019)
)

print(
    "Defaults excluded because not full-window:",
    len(all_90dpd_2019) - len(eligible_defaults_2019)
)

# ------------------------------------------------------------
# 9. Construct final target
# ------------------------------------------------------------

target_2019 = pd.DataFrame({
    "LoanSequenceNumber": full_window_loans_2019
})

target_2019["Default_36M"] = (
    target_2019["LoanSequenceNumber"]
    .isin(eligible_defaults_2019)
    .astype(int)
)

# ------------------------------------------------------------
# 10. Final target diagnostics
# ------------------------------------------------------------

print("\nTarget dataset shape:", target_2019.shape)

print("\nTarget distribution:")
print(
    target_2019["Default_36M"]
    .value_counts()
)

print("\nTarget proportions:")
print(
    target_2019["Default_36M"]
    .value_counts(normalize=True)
)

print(
    "\n2019 36-month default rate:",
    target_2019["Default_36M"].mean()
)

print(
    "\n2019 36-month default rate (%):",
    target_2019["Default_36M"].mean() * 100
)

In [ ]:
# ============================================================
# 2019 — MERGE ORIGINATION + 36M TARGET
# ============================================================

# Make absolutely sure both keys are strings
orig_2019["LoanSequenceNumber"] = (
    orig_2019["LoanSequenceNumber"]
    .astype(str)
    .str.strip()
)

target_2019["LoanSequenceNumber"] = (
    target_2019["LoanSequenceNumber"]
    .astype(str)
    .str.strip()
)

# Check uniqueness before merging
print(
    "Unique origination LoanSequenceNumber:",
    orig_2019["LoanSequenceNumber"].nunique()
)

print(
    "Unique target LoanSequenceNumber:",
    target_2019["LoanSequenceNumber"].nunique()
)

# Merge
model_2019 = orig_2019.merge(
    target_2019,
    on="LoanSequenceNumber",
    how="inner",
    validate="one_to_one"
)

print("\nModel dataset shape:", model_2019.shape)

print("\nTarget distribution:")
print(
    model_2019["Default_36M"]
    .value_counts()
)

print("\nTarget proportions:")
print(
    model_2019["Default_36M"]
    .value_counts(normalize=True)
)

print(
    "\nMissing target values:",
    model_2019["Default_36M"].isna().sum()
)

print("\nOriginal 2019 loans:", len(orig_2019))
print("Eligible modelling loans:", len(model_2019))

print(
    "Excluded from modelling:",
    len(orig_2019) - len(model_2019)
)

In [ ]:
# ============================================================
# 2019 — SPECIAL MISSING-VALUE / CODE AUDIT
# ============================================================

model_2019_clean = model_2019.copy()

print("Starting shape:", model_2019_clean.shape)

print("\nMissing values:")
print(
    model_2019_clean.isna()
    .sum()
    .sort_values(ascending=False)
)

print("\n--- SPECIAL CODES ---")

print(
    "CreditScore = 9999:",
    (model_2019_clean["CreditScore"] == 9999).sum()
)

print(
    "OriginalDTI = 999:",
    (model_2019_clean["OriginalDTI"] == 999).sum()
)

print(
    "OriginalLTV = 999:",
    (model_2019_clean["OriginalLTV"] == 999).sum()
)

print(
    "OriginalCLTV = 999:",
    (model_2019_clean["OriginalCLTV"] == 999).sum()
)

print(
    "OriginalInterestRate = 999:",
    (model_2019_clean["OriginalInterestRate"] == 999).sum()
)

print(
    "MIPercent missing:",
    model_2019_clean["MIPercent"].isna().sum()
)

In [ ]:
# ============================================================
# 2019 — CONVERT SPECIAL CODES
# ============================================================

# Credit score special code
model_2019_clean.loc[
    model_2019_clean["CreditScore"] == 9999,
    "CreditScore"
] = np.nan

# DTI special code
model_2019_clean.loc[
    model_2019_clean["OriginalDTI"] == 999,
    "OriginalDTI"
] = np.nan

# CLTV special code
model_2019_clean.loc[
    model_2019_clean["OriginalCLTV"] == 999,
    "OriginalCLTV"
] = np.nan

print("CreditScore missing after conversion:",
      model_2019_clean["CreditScore"].isna().sum())

print("DTI missing after conversion:",
      model_2019_clean["OriginalDTI"].isna().sum())

print("CLTV missing after conversion:",
      model_2019_clean["OriginalCLTV"].isna().sum())

# ------------------------------------------------------------
# Check whether these missing rows contain defaults
# ------------------------------------------------------------

for col in [
    "CreditScore",
    "OriginalDTI",
    "OriginalCLTV"
]:
    missing_rows = model_2019_clean[
        model_2019_clean[col].isna()
    ]

    print("\n", col)
    print("Missing rows:", len(missing_rows))
    print("Default distribution:")
    print(
        missing_rows["Default_36M"]
        .value_counts()
    )
    print(
        "Default rate:",
        missing_rows["Default_36M"].mean()
    )

In [ ]:
# ============================================================
# 2019 — FINAL VARIABLE CLEANUP
# ============================================================

drop_cols_2019 = [
    "MSA",
    "SuperConformingFlag",
    "PreReliefRefinanceLoanSequenceNumber"
]

model_2019_final = model_2019_clean.drop(
    columns=drop_cols_2019
).copy()

print("Final 2019 shape:", model_2019_final.shape)

print("\nRemaining missing values:")
print(
    model_2019_final.isna()
    .sum()
    .sort_values(ascending=False)
)

print("\nTarget distribution:")
print(
    model_2019_final["Default_36M"]
    .value_counts()
)

print("\nDefault rate:")
print(
    model_2019_final["Default_36M"].mean()
)

In [ ]:
# ============================================================
# FREDDIE MAC PROJECT — 2020 SAMPLE EXTRACTION
# ============================================================

import os
import glob
import zipfile
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. FIND 2020 ZIP
# ------------------------------------------------------------

downloads = r"./data/raw"

zip_candidates = [
    f for f in glob.glob(os.path.join(downloads, "*2020*.zip"))
    if "sample" in os.path.basename(f).lower()
]

print("2020 ZIP candidates:")
print([os.path.basename(x) for x in zip_candidates])

if len(zip_candidates) == 0:
    raise FileNotFoundError(
        "No 2020 sample ZIP found in Downloads."
    )

zip_path = zip_candidates[0]

print("\nUsing:")
print(zip_path)

# ------------------------------------------------------------
# 2. INSPECT ZIP
# ------------------------------------------------------------

with zipfile.ZipFile(zip_path, "r") as z:
    files_2020 = z.namelist()

print("\nFiles in ZIP:")
print(files_2020)

# ------------------------------------------------------------
# 3. EXTRACT
# ------------------------------------------------------------

extract_dir = os.path.join(downloads, "freddie_2020_extracted")
os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(extract_dir)

print("\nExtracted to:")
print(extract_dir)

# ------------------------------------------------------------
# 4. FIND ORIGINATION + PERFORMANCE FILES
# ------------------------------------------------------------

orig_candidates = [
    os.path.join(extract_dir, f)
    for f in os.listdir(extract_dir)
    if "orig" in f.lower() and f.lower().endswith(".txt")
]

perf_candidates = [
    os.path.join(extract_dir, f)
    for f in os.listdir(extract_dir)
    if "perf" in f.lower() and f.lower().endswith(".txt")
]

print("\nOrigination candidates:")
print([os.path.basename(x) for x in orig_candidates])

print("\nPerformance candidates:")
print([os.path.basename(x) for x in perf_candidates])

if not orig_candidates or not perf_candidates:
    raise FileNotFoundError(
        "Could not identify both origination and performance files."
    )

orig_path_2020 = orig_candidates[0]
perf_path_2020 = perf_candidates[0]

# ------------------------------------------------------------
# 5. READ RAW FILES
# ------------------------------------------------------------

orig_2020_raw = pd.read_csv(
    orig_path_2020,
    sep="|",
    header=None,
    low_memory=False
)

perf_2020_raw = pd.read_csv(
    perf_path_2020,
    sep="|",
    header=None,
    low_memory=False
)

print("\nOrigination shape:", orig_2020_raw.shape)
print("Performance shape:", perf_2020_raw.shape)

print("\nOrigination columns:", orig_2020_raw.shape[1])
print("Performance columns:", perf_2020_raw.shape[1])

# ------------------------------------------------------------
# 6. SHOW FIRST ROWS
# ------------------------------------------------------------

print("\nOrigination first row:")
print(orig_2020_raw.iloc[0].tolist())

print("\nPerformance first row:")
print(perf_2020_raw.iloc[0].tolist())

In [ ]:
# ============================================================
# 2020 — READ ORIGINATION + PERFORMANCE FILES
# ============================================================

import pandas as pd
import numpy as np
import os

base_2020 = r"./data/raw/freddie_2020_extracted"

orig_path_2020 = os.path.join(base_2020, "sample_orig_2020.txt")
perf_path_2020 = os.path.join(base_2020, "sample_perf_2020.txt")

# Read raw files
orig_2020_raw = pd.read_csv(
    orig_path_2020,
    sep="|",
    header=None,
    low_memory=False
)

perf_2020_raw = pd.read_csv(
    perf_path_2020,
    sep="|",
    header=None,
    low_memory=False
)

print("Origination shape:", orig_2020_raw.shape)
print("Performance shape:", perf_2020_raw.shape)

print("\nOrigination columns:", orig_2020_raw.shape[1])
print("Performance columns:", perf_2020_raw.shape[1])

print("\nOrigination first row:")
print(orig_2020_raw.iloc[0].tolist())

print("\nPerformance first row:")
print(perf_2020_raw.iloc[0].tolist())

In [ ]:
# ============================================================
# 2020 — MAP PERFORMANCE COLUMNS + CONSTRUCT LOAN AGE
# ============================================================

# Freddie Mac performance column positions
perf_columns = [
    "LoanSequenceNumber",
    "MonthlyReportingPeriod",
    "CurrentActualUPB",
    "LoanDelinquencyStatus",
    "LoanAge",
    "RemainingMonthsToLegalMaturity",
    "DefectSettlementDate",
    "ModificationFlag",
    "ZeroBalanceCode",
    "ZeroBalanceEffectiveDate",
    "CurrentInterestRate",
    "CurrentDeferredUPB",
    "DueDateOfLastPaidInstallment",
    "MIRecoveries",
    "NetSalesProceeds",
    "NonMIRecoveries",
    "Expenses",
    "LegalCosts",
    "MaintenanceAndPreservationCosts",
    "TaxesAndInsurance",
    "MiscellaneousExpenses",
    "ActualLossCalculation",
    "ModificationCost",
    "StepModificationFlag",
    "DeferredPaymentPlan",
    "EstimatedLoanToValue",
    "ZeroBalanceRemovalUPB",
    "DelinquentAccruedInterest",
    "RepurchaseMakeWholeProceeds",
    "CreditEventNetSaleProceeds",
    "HomeReadyProgramIndicator",
    "ReliefRefinanceIndicator",
    "InvestorLoanType",
    "PropertyValuationMethod",
    "InterestOnlyIndicator"
]

perf_2020 = perf_2020_raw.copy()
perf_2020.columns = perf_columns

print("Performance shape:", perf_2020.shape)

# ------------------------------------------------------------
# Convert LoanAge safely
# ------------------------------------------------------------

perf_2020["LoanAge"] = pd.to_numeric(
    perf_2020["LoanAge"],
    errors="coerce"
)

print("\nLoanAge range:")
print("Minimum:", perf_2020["LoanAge"].min())
print("Maximum:", perf_2020["LoanAge"].max())

print("\nLoanAge value counts:")
print(
    perf_2020["LoanAge"]
    .value_counts()
    .sort_index()
    .head(40)
)

In [ ]:
# ============================================================
# 2020 — CONSTRUCT 36-MONTH DEFAULT TARGET
# ============================================================

# Convert delinquency status to numeric
perf_2020["DelinqNumeric"] = pd.to_numeric(
    perf_2020["LoanDelinquencyStatus"],
    errors="coerce"
)

# ------------------------------------------------------------
# Restrict to first 36 months
# ------------------------------------------------------------

perf_2020_36m = perf_2020[
    perf_2020["LoanAge"].between(0, 36)
].copy()

print("Performance observations in 36M window:",
      len(perf_2020_36m))

print("\nLoanAge range in 36M window:")
print(
    perf_2020_36m["LoanAge"].min(),
    "to",
    perf_2020_36m["LoanAge"].max()
)

# ------------------------------------------------------------
# Identify loans with complete 36-month window
# ------------------------------------------------------------

months_per_loan_2020 = (
    perf_2020_36m
    .groupby("LoanSequenceNumber")["LoanAge"]
    .nunique()
)

full_window_loans_2020 = months_per_loan_2020[
    months_per_loan_2020 >= 37
].index

print("\nFull-window 2020 loans:",
      len(full_window_loans_2020))

# ------------------------------------------------------------
# Identify 90+ DPD within 36 months
# ------------------------------------------------------------

default_loans_2020 = (
    perf_2020_36m[
        perf_2020_36m["DelinqNumeric"] >= 3
    ]
    ["LoanSequenceNumber"]
    .drop_duplicates()
)

print(
    "All loans with 90+ DPD within 36 months:",
    len(default_loans_2020)
)

# ------------------------------------------------------------
# Keep only defaults among full-window loans
# ------------------------------------------------------------

eligible_default_loans_2020 = set(default_loans_2020).intersection(
    set(full_window_loans_2020)
)

excluded_defaults_2020 = (
    len(default_loans_2020)
    - len(eligible_default_loans_2020)
)

print(
    "Eligible 90+ DPD loans:",
    len(eligible_default_loans_2020)
)

print(
    "Defaults excluded because not full-window:",
    excluded_defaults_2020
)

# ------------------------------------------------------------
# Construct target for ALL full-window loans
# ------------------------------------------------------------

target_2020 = pd.DataFrame({
    "LoanSequenceNumber": list(full_window_loans_2020)
})

target_2020["Default_36M"] = (
    target_2020["LoanSequenceNumber"]
    .isin(eligible_default_loans_2020)
    .astype(int)
)

print("\nTarget dataset shape:", target_2020.shape)

print("\nTarget distribution:")
print(
    target_2020["Default_36M"]
    .value_counts()
)

print("\nTarget proportions:")
print(
    target_2020["Default_36M"]
    .value_counts(normalize=True)
)

print(
    "\n2020 36-month default rate:",
    target_2020["Default_36M"].mean()
)

print(
    "2020 36-month default rate (%):",
    target_2020["Default_36M"].mean() * 100
)

In [ ]:
# ============================================================
# 2020 — CORRECT ORIGINATION COLUMN MAPPING
# ============================================================

orig_columns_2020 = [
    "CreditScore",
    "FirstPaymentDate",
    "FirstTimeHomebuyerFlag",
    "MaturityDate",
    "MSA",
    "MIPercent",
    "NumberOfUnits",
    "OccupancyStatus",
    "OriginalCLTV",
    "OriginalDTI",
    "OriginalUPB",
    "OriginalLTV",
    "OriginalInterestRate",
    "Channel",
    "PPMFlag",
    "AmortizationType",
    "PropertyState",
    "PropertyType",
    "PostalCode",
    "LoanSequenceNumber",
    "LoanPurpose",
    "OriginalLoanTerm",
    "NumberOfBorrowers",
    "SellerName",
    "ServicerName",
    "SuperConformingFlag",
    "PreReliefRefinanceLoanSequenceNumber",
    "SpecialEligibilityProgram",
    "ReliefRefinanceIndicator",
    "PropertyValuationMethod",
    "InterestOnlyIndicator",
    "UnusedField"
]

print("Raw columns:", orig_2020_raw.shape[1])
print("Names supplied:", len(orig_columns_2020))

In [ ]:
orig_columns_2020 = [
    "CreditScore",
    "FirstPaymentDate",
    "FirstTimeHomebuyerFlag",
    "MaturityDate",
    "MSA",
    "MIPercent",
    "NumberOfUnits",
    "OccupancyStatus",
    "OriginalCLTV",
    "OriginalDTI",
    "OriginalUPB",
    "OriginalLTV",
    "OriginalInterestRate",
    "Channel",
    "PPMFlag",
    "AmortizationType",
    "PropertyState",
    "PropertyType",
    "PostalCode",
    "LoanSequenceNumber",
    "LoanPurpose",
    "OriginalLoanTerm",
    "NumberOfBorrowers",
    "SellerName",
    "ServicerName",
    "SuperConformingFlag",
    "PreReliefRefinanceLoanSequenceNumber",
    "SpecialEligibilityProgram",
    "ReliefRefinanceIndicator",
    "PropertyValuationMethod",
    "InterestOnlyIndicator"
]

print("Raw columns:", orig_2020_raw.shape[1])
print("Names supplied:", len(orig_columns_2020))

In [ ]:
# ============================================================
# APPLY 2020 ORIGINATION COLUMN NAMES
# ============================================================

orig_2020 = orig_2020_raw.copy()
orig_2020.columns = orig_columns_2020

print("Origination shape:", orig_2020.shape)

print("\nFirst row:")
print(orig_2020.iloc[0].to_dict())

print("\nUnique LoanSequenceNumber:",
      orig_2020["LoanSequenceNumber"].nunique())

print("Missing LoanSequenceNumber:",
      orig_2020["LoanSequenceNumber"].isna().sum())

In [ ]:
# ============================================================
# 2020 — MERGE ORIGINATION + 36M TARGET
# ============================================================

orig_2020["LoanSequenceNumber"] = (
    orig_2020["LoanSequenceNumber"]
    .astype(str)
    .str.strip()
)

target_2020["LoanSequenceNumber"] = (
    target_2020["LoanSequenceNumber"]
    .astype(str)
    .str.strip()
)

print("Unique origination IDs:",
      orig_2020["LoanSequenceNumber"].nunique())

print("Unique target IDs:",
      target_2020["LoanSequenceNumber"].nunique())

model_2020 = orig_2020.merge(
    target_2020,
    on="LoanSequenceNumber",
    how="inner",
    validate="one_to_one"
)

print("\nModel dataset shape:", model_2020.shape)

print("\nTarget distribution:")
print(model_2020["Default_36M"].value_counts())

print("\nTarget proportions:")
print(model_2020["Default_36M"].value_counts(normalize=True))

print("\nMissing target values:",
      model_2020["Default_36M"].isna().sum())

print("\nOriginal 2020 loans:", len(orig_2020))
print("Eligible modelling loans:", len(model_2020))
print("Excluded from modelling:",
      len(orig_2020) - len(model_2020))

In [ ]:
# ============================================================
# 2020 — SPECIAL CODE / MISSING-VALUE AUDIT
# ============================================================

print("Starting shape:", model_2020.shape)

print("\nMissing values:")
print(
    model_2020.isna()
    .sum()
    .sort_values(ascending=False)
)

print("\n--- SPECIAL CODES ---")

print(
    "CreditScore = 9999:",
    (model_2020["CreditScore"] == 9999).sum()
)

print(
    "OriginalDTI = 999:",
    (model_2020["OriginalDTI"] == 999).sum()
)

print(
    "OriginalLTV = 999:",
    (model_2020["OriginalLTV"] == 999).sum()
)

print(
    "OriginalCLTV = 999:",
    (model_2020["OriginalCLTV"] == 999).sum()
)

print(
    "OriginalInterestRate = 999:",
    (model_2020["OriginalInterestRate"] == 999).sum()
)

print(
    "MIPercent missing:",
    model_2020["MIPercent"].isna().sum()
)

In [ ]:
# ============================================================
# 2020 — SPECIAL MISSING-VALUE CONVERSION
# ============================================================

model_2020_clean = model_2020.copy()

# Convert Freddie Mac special missing code
model_2020_clean.loc[
    model_2020_clean["CreditScore"] == 9999,
    "CreditScore"
] = np.nan

print("CreditScore missing after conversion:",
      model_2020_clean["CreditScore"].isna().sum())

print("\nCreditScore-missing default distribution:")
print(
    model_2020_clean.loc[
        model_2020_clean["CreditScore"].isna(),
        "Default_36M"
    ].value_counts()
)

print("\nCreditScore-missing default rate:")
print(
    model_2020_clean.loc[
        model_2020_clean["CreditScore"].isna(),
        "Default_36M"
    ].mean()
)

In [ ]:
# ============================================================
# 2020 — DROP UNUSABLE VARIABLES
# ============================================================

drop_cols_2020 = [
    "MSA",
    "SuperConformingFlag",
    "PreReliefRefinanceLoanSequenceNumber"
]

model_2020_final = model_2020_clean.drop(
    columns=drop_cols_2020
)

print("Final 2020 shape:", model_2020_final.shape)

print("\nRemaining missing values:")
print(
    model_2020_final.isna()
    .sum()
    .sort_values(ascending=False)
)

In [ ]:
# ============================================================
# FREDDIE MAC — 2022 SAMPLE EXTRACTION
# ============================================================

import os
import zipfile
import glob
import pandas as pd
import numpy as np

zip_path = r"./data/raw/sample_2022.zip"

print("Using:")
print(zip_path)

if not os.path.exists(zip_path):
    raise FileNotFoundError(
        f"2022 ZIP not found: {zip_path}"
    )

extract_dir = r"./data/raw/freddie_2022_extracted"

os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as z:
    print("\nFiles in ZIP:")
    print(z.namelist())
    
    z.extractall(extract_dir)

print("\nExtracted to:")
print(extract_dir)

# Find files
orig_candidates = glob.glob(
    os.path.join(extract_dir, "*orig*2022*.txt")
)

perf_candidates = glob.glob(
    os.path.join(extract_dir, "*perf*2022*.txt")
)

print("\nOrigination candidates:")
print([os.path.basename(f) for f in orig_candidates])

print("\nPerformance candidates:")
print([os.path.basename(f) for f in perf_candidates])

In [ ]:
# ============================================================
# READ 2022 RAW FILES
# ============================================================

orig_2022_raw = pd.read_csv(
    orig_candidates[0],
    sep="|",
    header=None,
    low_memory=False
)

perf_2022_raw = pd.read_csv(
    perf_candidates[0],
    sep="|",
    header=None,
    low_memory=False
)

print("Origination shape:", orig_2022_raw.shape)
print("Performance shape:", perf_2022_raw.shape)

print("\nOrigination columns:", orig_2022_raw.shape[1])
print("Performance columns:", perf_2022_raw.shape[1])

print("\nOrigination first row:")
print(orig_2022_raw.iloc[0].tolist())

print("\nPerformance first row:")
print(perf_2022_raw.iloc[0].tolist())

In [ ]:
# ============================================================
# 2022 — ASSIGN ORIGINATION COLUMN NAMES
# ============================================================

orig_columns = [
    "CreditScore",
    "FirstPaymentDate",
    "FirstTimeHomebuyerFlag",
    "MaturityDate",
    "MSA",
    "MIPercent",
    "NumberOfUnits",
    "OccupancyStatus",
    "OriginalCLTV",
    "OriginalDTI",
    "OriginalUPB",
    "OriginalLTV",
    "OriginalInterestRate",
    "Channel",
    "PPMFlag",
    "AmortizationType",
    "PropertyState",
    "PropertyType",
    "PostalCode",
    "LoanSequenceNumber",
    "LoanPurpose",
    "OriginalLoanTerm",
    "NumberOfBorrowers",
    "SellerName",
    "ServicerName",
    "SuperConformingFlag",
    "PreReliefRefinanceLoanSequenceNumber",
    "SpecialEligibilityProgram",
    "ReliefRefinanceIndicator",
    "PropertyValuationMethod",
    "InterestOnlyIndicator"
]

print("Raw columns:", orig_2022_raw.shape[1])
print("Names supplied:", len(orig_columns))

if orig_2022_raw.shape[1] != len(orig_columns):
    raise ValueError(
        f"Expected 31 columns, got {orig_2022_raw.shape[1]}"
    )

orig_2022 = orig_2022_raw.copy()
orig_2022.columns = orig_columns

print("\nOrigination shape:", orig_2022.shape)

print("\nFirst row:")
print(orig_2022.iloc[0].to_dict())

print("\nUnique LoanSequenceNumber:",
      orig_2022["LoanSequenceNumber"].nunique())

print("Missing LoanSequenceNumber:",
      orig_2022["LoanSequenceNumber"].isna().sum())

In [ ]:
# ============================================================
# FREDDIE MAC 2022 — CONSTRUCT 36-MONTH DEFAULT TARGET
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. STANDARD FREDDIE MAC PERFORMANCE COLUMN NAMES
# ------------------------------------------------------------

perf_columns = [
    "LoanSequenceNumber",
    "MonthlyReportingPeriod",
    "CurrentActualUPB",
    "CurrentLoanDelinquencyStatus",
    "LoanAge",
    "RemainingMonthsToLegalMaturity",
    "DefectSettlementDate",
    "ModificationFlag",
    "ZeroBalanceCode",
    "ZeroBalanceEffectiveDate",
    "CurrentInterestRate",
    "CurrentDeferredUPB",
    "DueDate",
    "LastPaidInstallmentDate",
    "MIRecoveries",
    "NetSalesProceeds",
    "NonMIRecoveries",
    "Expenses",
    "LegalCosts",
    "MaintenanceAndPreservationCosts",
    "TaxesAndInsurance",
    "MiscellaneousExpenses",
    "ActualLossCalculation",
    "ModificationCost",
    "StepModificationFlag",
    "DeferredPaymentPlan",
    "EstimatedLoanToValue",
    "ZeroBalanceRemovalUPB",
    "DelinquentAccruedInterest",
    "PropertyInspectionWaivedIndicator",
    "ForeclosureCosts",
    "PropertyDispositionDate",
    "ForeclosureDate",
    "REOAdditionalProceeds",
    "UnknownField"
]

print("Raw performance columns:", perf_2022_raw.shape[1])
print("Names supplied:", len(perf_columns))

# Safety check
if perf_2022_raw.shape[1] != len(perf_columns):
    raise ValueError(
        f"Performance column mismatch: "
        f"{perf_2022_raw.shape[1]} raw columns vs "
        f"{len(perf_columns)} supplied names"
    )

perf_2022 = perf_2022_raw.copy()
perf_2022.columns = perf_columns

print("Performance shape:", perf_2022.shape)

# ------------------------------------------------------------
# 2. CLEAN LOAN AGE
# ------------------------------------------------------------

perf_2022["LoanAge"] = pd.to_numeric(
    perf_2022["LoanAge"],
    errors="coerce"
)

# Keep only sensible integer loan ages
perf_2022 = perf_2022[
    perf_2022["LoanAge"].between(0, 100)
].copy()

perf_2022["LoanAge"] = perf_2022["LoanAge"].astype(int)

print("\nCorrect LoanAge range:")
print("Minimum:", perf_2022["LoanAge"].min())
print("Maximum:", perf_2022["LoanAge"].max())

# ------------------------------------------------------------
# 3. KEEP FIRST 36 MONTHS
# ------------------------------------------------------------

perf_2022_36m = perf_2022[
    perf_2022["LoanAge"].between(0, 36)
].copy()

print("\nPerformance observations in 36M window:",
      len(perf_2022_36m))

print("LoanAge range in 36M window:")
print(
    perf_2022_36m["LoanAge"].min(),
    "to",
    perf_2022_36m["LoanAge"].max()
)

# ------------------------------------------------------------
# 4. IDENTIFY FULL-WINDOW LOANS
# ------------------------------------------------------------

loan_age_coverage_2022 = (
    perf_2022_36m
    .groupby("LoanSequenceNumber")["LoanAge"]
    .agg(["min", "max", "nunique"])
)

full_window_ids_2022 = loan_age_coverage_2022[
    (loan_age_coverage_2022["min"] == 0) &
    (loan_age_coverage_2022["max"] >= 36)
].index

print("\nFull-window 2022 loans:",
      len(full_window_ids_2022))

# ------------------------------------------------------------
# 5. CONVERT DELINQUENCY STATUS
# ------------------------------------------------------------

perf_2022_36m["DelinqNumeric"] = pd.to_numeric(
    perf_2022_36m["CurrentLoanDelinquencyStatus"],
    errors="coerce"
)

# ------------------------------------------------------------
# 6. IDENTIFY 90+ DPD
# ------------------------------------------------------------

default_loans_2022 = (
    perf_2022_36m.loc[
        perf_2022_36m["DelinqNumeric"] >= 3,
        "LoanSequenceNumber"
    ]
    .dropna()
    .unique()
)

print("\nAll loans with 90+ DPD within 36 months:",
      len(default_loans_2022))

# ------------------------------------------------------------
# 7. RETAIN ONLY FULL-WINDOW LOANS
# ------------------------------------------------------------

eligible_default_ids_2022 = set(default_loans_2022).intersection(
    set(full_window_ids_2022)
)

print("Eligible 90+ DPD loans:",
      len(eligible_default_ids_2022))

print(
    "Defaults excluded because not full-window:",
    len(set(default_loans_2022) - set(full_window_ids_2022))
)

# ------------------------------------------------------------
# 8. BUILD TARGET DATASET
# ------------------------------------------------------------

target_2022 = pd.DataFrame({
    "LoanSequenceNumber": list(full_window_ids_2022)
})

target_2022["Default_36M"] = (
    target_2022["LoanSequenceNumber"]
    .isin(eligible_default_ids_2022)
    .astype(int)
)

print("\nTarget dataset shape:", target_2022.shape)

print("\nTarget distribution:")
print(target_2022["Default_36M"].value_counts())

print("\nTarget proportions:")
print(target_2022["Default_36M"].value_counts(normalize=True))

print(
    "\n2022 36-month default rate:",
    target_2022["Default_36M"].mean()
)

print(
    "2022 36-month default rate (%):",
    target_2022["Default_36M"].mean() * 100
)

In [ ]:
# ============================================================
# 2022 — MERGE ORIGINATION + 36M TARGET
# ============================================================

# Make sure both keys are strings
orig_2022["LoanSequenceNumber"] = (
    orig_2022["LoanSequenceNumber"]
    .astype(str)
    .str.strip()
)

target_2022["LoanSequenceNumber"] = (
    target_2022["LoanSequenceNumber"]
    .astype(str)
    .str.strip()
)

print("Unique origination IDs:",
      orig_2022["LoanSequenceNumber"].nunique())

print("Unique target IDs:",
      target_2022["LoanSequenceNumber"].nunique())

# Check uniqueness before merge
print("\nDuplicate origination IDs:",
      orig_2022["LoanSequenceNumber"].duplicated().sum())

print("Duplicate target IDs:",
      target_2022["LoanSequenceNumber"].duplicated().sum())

# Merge
model_2022 = orig_2022.merge(
    target_2022,
    on="LoanSequenceNumber",
    how="inner",
    validate="one_to_one"
)

print("\nModel dataset shape:", model_2022.shape)

print("\nTarget distribution:")
print(model_2022["Default_36M"].value_counts())

print("\nTarget proportions:")
print(model_2022["Default_36M"].value_counts(normalize=True))

print("\nMissing target values:",
      model_2022["Default_36M"].isna().sum())

print("\nOriginal 2022 loans:", len(orig_2022))
print("Eligible modelling loans:", len(model_2022))
print("Excluded from modelling:",
      len(orig_2022) - len(model_2022))

In [ ]:
# ============================================================
# 2022 — CHECK SPECIAL MISSING-VALUE CODES
# ============================================================

print("CreditScore = 9999:",
      (model_2022["CreditScore"] == 9999).sum())

print("OriginalDTI = 999:",
      (model_2022["OriginalDTI"] == 999).sum())

print("OriginalLTV = 999:",
      (model_2022["OriginalLTV"] == 999).sum())

print("OriginalCLTV = 999:",
      (model_2022["OriginalCLTV"] == 999).sum())

print("OriginalInterestRate = 999:",
      (model_2022["OriginalInterestRate"] == 999).sum())

print("MIPercent missing:",
      model_2022["MIPercent"].isna().sum())

print("\nMissing values:")
print(
    model_2022.isna()
    .sum()
    .sort_values(ascending=False)
)

In [ ]:
# ============================================================
# 2022 — SPECIAL MISSING-VALUE CONVERSION
# ============================================================

model_2022_clean = model_2022.copy()

# Freddie Mac special missing-value codes
model_2022_clean.loc[
    model_2022_clean["CreditScore"] == 9999,
    "CreditScore"
] = np.nan

model_2022_clean.loc[
    model_2022_clean["OriginalDTI"] == 999,
    "OriginalDTI"
] = np.nan

print("CreditScore missing after conversion:",
      model_2022_clean["CreditScore"].isna().sum())

print("DTI missing after conversion:",
      model_2022_clean["OriginalDTI"].isna().sum())

print("\nCreditScore-missing default distribution:")
print(
    model_2022_clean.loc[
        model_2022_clean["CreditScore"].isna(),
        "Default_36M"
    ].value_counts()
)

print("\nDTI-missing default distribution:")
print(
    model_2022_clean.loc[
        model_2022_clean["OriginalDTI"].isna(),
        "Default_36M"
    ].value_counts()
)

# ============================================================
# DROP VARIABLES THAT ARE UNUSABLE
# ============================================================

drop_cols_2022 = [
    "MSA",
    "SuperConformingFlag",
    "PreReliefRefinanceLoanSequenceNumber"
]

model_2022_final = model_2022_clean.drop(
    columns=drop_cols_2022
)

print("\nFinal 2022 shape:", model_2022_final.shape)

print("\nRemaining missing values:")
print(
    model_2022_final.isna()
    .sum()
    .sort_values(ascending=False)
)

print("\nTarget distribution:")
print(
    model_2022_final["Default_36M"].value_counts()
)

print("\nDefault rate:")
print(
    model_2022_final["Default_36M"].mean()
)

In [ ]:
# ============================================================
# FREDDIE MAC PD PROJECT
# MASTER DATASET — 2018 TO 2022
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. ADD VINTAGE IDENTIFIER
# ------------------------------------------------------------

df_2018 = model_2018_final.copy()
df_2019 = model_2019_final.copy()
df_2020 = model_2020_final.copy()
df_2021 = model_2021_final.copy()
df_2022 = model_2022_final.copy()

df_2018["Vintage"] = 2018
df_2019["Vintage"] = 2019
df_2020["Vintage"] = 2020
df_2021["Vintage"] = 2021
df_2022["Vintage"] = 2022

# ------------------------------------------------------------
# 2. COMBINE ALL VINTAGES
# ------------------------------------------------------------

master_pd = pd.concat(
    [
        df_2018,
        df_2019,
        df_2020,
        df_2021,
        df_2022
    ],
    ignore_index=True
)

print("MASTER DATASET SHAPE:")
print(master_pd.shape)

# ------------------------------------------------------------
# 3. CHECK VINTAGE DISTRIBUTION
# ------------------------------------------------------------

print("\nLOANS BY VINTAGE:")
print(
    master_pd["Vintage"]
    .value_counts()
    .sort_index()
)

# ------------------------------------------------------------
# 4. CHECK DEFAULT DISTRIBUTION
# ------------------------------------------------------------

print("\nDEFAULT DISTRIBUTION:")
print(
    master_pd["Default_36M"]
    .value_counts()
)

print("\nDEFAULT PROPORTIONS:")
print(
    master_pd["Default_36M"]
    .value_counts(normalize=True)
)

print("\nOVERALL DEFAULT RATE:")
print(
    master_pd["Default_36M"].mean()
)

# ------------------------------------------------------------
# 5. DEFAULT RATE BY VINTAGE
# ------------------------------------------------------------

print("\nDEFAULT RATE BY VINTAGE:")

vintage_default = (
    master_pd
    .groupby("Vintage")["Default_36M"]
    .agg(
        Loans="count",
        Defaults="sum",
        DefaultRate="mean"
    )
)

vintage_default["DefaultRatePct"] = (
    vintage_default["DefaultRate"] * 100
)

print(vintage_default)

# ------------------------------------------------------------
# 6. DUPLICATE LOAN IDS
# ------------------------------------------------------------

print("\nDUPLICATE LOAN IDS:")

duplicate_ids = (
    master_pd["LoanSequenceNumber"]
    .duplicated()
    .sum()
)

print(duplicate_ids)

# ------------------------------------------------------------
# 7. MISSING TARGET
# ------------------------------------------------------------

print("\nMISSING TARGET VALUES:")
print(
    master_pd["Default_36M"].isna().sum()
)

# ------------------------------------------------------------
# 8. MISSING VALUES
# ------------------------------------------------------------

print("\nMISSING VALUES:")
missing_master = (
    master_pd
    .isna()
    .sum()
    .sort_values(ascending=False)
)

print(missing_master[missing_master > 0])

# ------------------------------------------------------------
# 9. DATA TYPES
# ------------------------------------------------------------

print("\nDATA TYPES:")
print(master_pd.dtypes)

# ------------------------------------------------------------
# 10. FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("MASTER PD DATASET SUMMARY")
print("=" * 60)

print("Total loans:", len(master_pd))
print("Total defaults:", master_pd["Default_36M"].sum())
print(
    "Overall default rate:",
    round(master_pd["Default_36M"].mean() * 100, 3),
    "%"
)
print(
    "Number of vintages:",
    master_pd["Vintage"].nunique()
)
print(
    "Duplicate loan IDs:",
    duplicate_ids
)
print(
    "Missing target values:",
    master_pd["Default_36M"].isna().sum()
)

In [ ]:
# ============================================================
# SAVE MASTER PD DATASET FOR SQL
# ============================================================

output_path = r"./data/raw/freddie_master_pd_2018_2022.csv"

master_pd.to_csv(
    output_path,
    index=False
)

print("CSV saved successfully!")
print(output_path)
print("Shape:", master_pd.shape)

In [ ]:
print(list(master_pd.columns))